# 审核版本表（模板格式；内容级 SHA-256 由单元 1 现场计算复核）

| 审核版本 | 组 | 提交 SHA（提交级） | 文件（内容级 SHA-256，前 8 位，全文见单元 1 输出） |
| --- | --- | --- | --- |
| A1 正文/实验/习题与解答/编写报告 | G8 | main `b668d9e45bd1af2500f8d8849cb8292b8b452a29` | `A1-G8-正文.md` `06AA6B3C…`；`A1组-实验.ipynb` `D87F511F…`；`A1组-习题.md` `9EB55AC6…`；`A1组-习题参考解答.md` `13FC3597…`；`A1组-编写报告.md` `EA1FCB7C…` |
| A2 正文/实验/习题与解答/编写报告 | G21 | main `b668d9e4…`；B1 丁所审的 `230873df` 版本与 main 只差图号重排与小结标题（本组逐行 diff，§2.3–§2.4 无数学变化） | `A2-G21-正文.md` `EEACE11E…`；`A2组-实验.ipynb` `C6B4DB08…`；`A2组-习题.md` `B012AD30…`；`A2组-习题参考解答.md` `ED4EB66D…`；`A2组-编写报告.md` `E1199FFF…` |
| B1 验证报告/修正清单/验证 Notebook/AI 记录 | G12 | main `b668d9e4…`；B1 自述固定版本与本组核对结果见[批判报告"审核版本"](C1组-批判报告.md) | `B1组-验证报告.md` `D1F90AE8…`；`B1组-修正清单.md` `1F2D625C…`；`B1组-验证.ipynb` `0A47962F…`；`B1组-AI对话记录.md` `A66FF4A8…` |
| B2 | G22 | **2026-11-11 已提交**（main `c328c14`）；初稿时的事实记录：截至 2026-11-10 三处核查均无 B2 提交，当时标待复核 | `B2组-验证报告.md` `5369F021…`；`B2组-修正清单.md` `8A6D1E90…`；`B2组-验证.ipynb` `6BD66DCB…`；`B2组-AI对话记录.md` `463AD252…`；本组已交叉复核其 B001–B009（见单元 8），其被审 A1/A2 与本组所审 main **逐字节相同** |

# 第02章 逼近的智慧 · C1组（G6）人工复核

- 本章组名 / 实际组别：C1组 = G6；角色：C 批判
- 轮次：第02章第一轮批判与复核（针对 10 月 31 日初稿及其后续修订）
- 日期：2026-11-10
- 被审基准：仓库 `main` 最新提交 `c328c14`（2026-11-11）下 `教材/第02章-逼近的智慧/工作/` 的 A1、A2、B1、**B2** 交付物；A1/A2/B1 各文件 SHA-256 与本组 2026-11-10 审计时（`b668d9e45bd1af2500f8d8849cb8292b8b452a29`）**逐字节相同**，B2 为本轮新增提交。逐文件 SHA-256 由本笔记本第一个代码单元现场计算，不抄录他人记录。
- 独立性说明：本笔记本的每项判断都从题设或 A1/A2 程序的公开定义出发，由本组重新推导、重新实现后运行得到；**未使用本章另一 C 组的材料，也不以任何 AI 输出作为结论依据**。代码来源在每个单元内注明（本组手写 / 参照 A 组脚本的定义重写）。
- 编号：批判问题 `CH02-G6-C001—CH02-G6-C017`，AI 陷阱 `CH02-G6-T001—CH02-G6-T005`，见[批判报告](C1组-批判报告.md)、[AI陷阱](C1组-AI陷阱.md)、[手动推导](C1组-手动推导.md)。
- 判断口径：**已确认**（本组有推导或实测支撑）/ **不成立** / **证据不足**（无法定义或复现，需 A 组说明）。

In [1]:
import hashlib, json, sys, math, platform, os
import numpy as np
from mpmath import mp, mpf, zeta

mp.dps = 50
from pathlib import Path
def locate_work():
    # 按目录名定位本章 工作/，不写死机器路径；找不到时退回作者本机路径并如实打印
    target = ("教材", "第02章-逼近的智慧", "工作")
    seeds = [Path.cwd(), *Path.cwd().resolve().parents]
    seeds.append(Path.home() / "Desktop" / "科学计算" / "课程仓库")
    for base in seeds:
        cand = base.joinpath(*target)
        if (cand / "A1-G8-正文.md").is_file():
            return str(cand) + "/"
    # 本交付文件夹与课程仓库同级的布局：尝试同级 "课程仓库" 目录
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        cand = base / "课程仓库" / "教材" / "第02章-逼近的智慧" / "工作"
        if (cand / "A1-G8-正文.md").is_file():
            return str(cand) + "/"
    return None
W = locate_work()
if W is None:
    W = "C:/Users/trw/Desktop/科学计算/课程仓库/教材/第02章-逼近的智慧/工作/"
    print("警告：未能按目录名定位本章 工作/，回退到作者本机绝对路径；若在他人环境运行，请把 Notebook 放在本章目录内再执行。")
print("被审材料目录 =", W)
AUDIT = ["A1-G8-正文.md", "A2-G21-正文.md", "A1组-实验.ipynb", "A2组-实验.ipynb",
         "A1组-习题.md", "A1组-习题参考解答.md", "A2组-习题.md", "A2组-习题参考解答.md",
         "A1组-编写报告.md", "A2组-编写报告.md",
         "B1组-验证报告.md", "B1组-修正清单.md", "B1组-验证.ipynb", "B1组-AI对话记录.md",
         "B2组-验证报告.md", "B2组-修正清单.md", "B2组-验证.ipynb", "B2组-AI对话记录.md",
         "G8-附件/ch2_numerics.py", "G8-附件/实验结果.json", "G8-附件/审校/独立数值补核.json"]
print("内核:", sys.version.split()[0], "| numpy", np.__version__, "| mpmath", __import__("mpmath").__version__, "|", platform.platform())
print("依赖：numpy + mpmath（scipy/matplotlib/torch 均不需要）。本内核无 PyTorch：A2 的 torch 相关主张改由本组自编反向模式 AD、复步长差分、mpmath 高精度数值微分三条独立途径核对。")
print("\n被审文件 SHA-256（现场计算，前16位）")
for f in AUDIT:
    p = W + f
    print("  ", hashlib.sha256(open(p, "rb").read()).hexdigest()[:16] if os.path.exists(p) else "缺失", f)
b2files = sorted(f for f in os.listdir(W) if f.startswith("B2组"))
print("\nB2组(G22) 本轮交付物:", b2files)
RES = json.load(open(W + "G8-附件/实验结果.json", encoding="utf-8"))
HP = json.load(open(W + "G8-附件/审校/独立数值补核.json", encoding="utf-8"))
print("A1 高精度补核记录:", {k: (v[:24] + "..." if isinstance(v, str) else v) for k, v in HP.items()})
EPS = np.finfo(float).eps

被审材料目录 = C:\Users\trw\Desktop\科学计算\课程仓库\教材\第02章-逼近的智慧\工作/
内核: 3.11.11 | numpy 1.26.4 | mpmath 1.3.0 | Windows-10-10.0.26200-SP0
依赖：numpy + mpmath（scipy/matplotlib/torch 均不需要）。本内核无 PyTorch：A2 的 torch 相关主张改由本组自编反向模式 AD、复步长差分、mpmath 高精度数值微分三条独立途径核对。

被审文件 SHA-256（现场计算，前16位）
   06aa6b3c0d6259eb A1-G8-正文.md
   eeace11e2e6684ca A2-G21-正文.md
   d87f511fb7296d9e A1组-实验.ipynb
   c6b4db08bd1a9fcb A2组-实验.ipynb
   9eb55ac69facf023 A1组-习题.md
   13fc3597a019bd77 A1组-习题参考解答.md
   b012ad30f23bf9b3 A2组-习题.md
   ed4eb66d627cb5b7 A2组-习题参考解答.md
   ea1fcb7cb8722639 A1组-编写报告.md
   e1199fffebf425d3 A2组-编写报告.md
   d1f90ae8dbca7fa8 B1组-验证报告.md
   1f2d625ca85bd19a B1组-修正清单.md
   0a47962f73cfb98c B1组-验证.ipynb
   a66ff4a8816d9c49 B1组-AI对话记录.md
   5369f0212dcb990b B2组-验证报告.md
   8a6d1e90f143b714 B2组-修正清单.md
   6bd66dcba2282442 B2组-验证.ipynb
   463ad252b8059983 B2组-AI对话记录.md
   9b167046bdf5fb2d G8-附件/ch2_numerics.py
   df51688fb3f4a21d G8-附件/实验结果.json
   ff577e9c81a02b27 G8-附件/审校/独立数值补核.json

B2组(G22) 本轮交付物: ['B2组-A

## 2.1 插值与Runge现象（2学时）

> 模板核对表：每行的"手动结果"见[手动推导](C1组-手动推导.md)，"数值结果"见下方单元的真实输出。

| 待核对主张 | 验证方法与来源 | 手动结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 定理 2.1.1/2.1.3/2.1.4、式(2.1.5)(2.1.8)；A2 式(2.1-6) | 手动推导 §1；单元 1–3 | 逐条独立重证，一致；补一般区间完整式 | 9 节点等距 1.0452、Chebyshev 0.17084 逐位复现；六档网格最大误差与 A1 记录一致 | 成立 |
| A1 §2.1.3 末重心权重括注 | 手动推导 §2；单元 2 | 零点族 (-1)^j·sinθ_j 正确；极值点族本组补全为 (-1)^j·δ_j | 两族与连乘定义偏差 6.7e-16 | 成立；建议补出极值点族并说明端点 1/2 |
| A1 用 Λ_n 解释 Runge 但"不展开证明"，并把 5.91e-9 只记作"量级可信" | 手动推导 §3；单元 1、26 | Λ_n·ε 控制扰动放大；Λ_32=2.43e7 | Λ_32·ε=5.40e-9 与 A1 记录差 9.6%；对抗数据放大/Λ=0.99999999 | CH02-G6-C005 已确认（改法：补 Λ 表并改写该句） |
| A1 附录 A.3 "（除非漏掉峰值）" | 单元 1 | 网格最大误差恒 ≤ 连续上确界；漏峰只偏小 | 三组加密单调不减、从不越界（比值 →1.00000000） | CH02-G6-C001 已确认 |
| "该换的是节点"作为唯一对策 | 手动推导 §10；单元 1 末 | 分段线性 ‖f−I_hf‖ ≤ M_2·h²/8，节点是否等距无关 | 33 个同样等距节点：分段线性 0.0207（界 0.0244） vs 全局 5059 | CH02-G6-C006 已确认（建议补分段对策一段） |

## 单元 1（2.1 插值与 Runge 现象）

**待核主张**

1. A1 式(2.1.5)/(2.1.8)：误差 = 函数因子 × 几何因子，Chebyshev 零点使几何因子取最小 $2^{-n}$。
2. A1"扩展·龙格现象"：等距节点 Lebesgue 常数随 $n$ 指数增长、Chebyshev 零点只随 $\log n$ 增长，"本节不展开证明"；并把 90 位复算与 float64 的相对差 $5.91\times10^{-9}$ 记作"量级可信"。
3. A1 附录 A.3：网格最大误差"不会超过连续范数（除非漏掉峰值）"。
4. A2 §2.1：不得由节点乘积或误差上界反推实际发散。
5. 龙格现象的教学对策是否唯一（A1："该换的是节点"）。

**代码来源**：重心求值与 Lebesgue 函数由本组按式(2.1.1)–(2.1.3) 自行实现；节点族与评价网格（6001/400001 点、偏移 0.371）按 A1 正文 §2.1 实验表的定义重写，不复用 A1 的函数。

In [2]:
def feq(x):
    return 1.0 / (1.0 + 25.0 * x * x)

def nodes_equi(m):
    return np.linspace(-1.0, 1.0, m)

def nodes_cheb(m):                      # T_m 的 m 个零点（式 2.1.7）
    return np.cos((2 * np.arange(m) + 1) * np.pi / (2 * m))

def bary_w(x):
    d = x[:, None] - x[None, :]
    np.fill_diagonal(d, 1.0)
    lw = -np.sum(np.log(np.abs(d)), axis=1)
    return np.prod(np.sign(d), axis=1) * np.exp(lw - lw.max())

def bary_eval(xs, ys, z, w=None):
    w = bary_w(xs) if w is None else w
    dd = z[:, None] - xs[None, :]
    hit = dd == 0
    t = w / np.where(hit, 1.0, dd)
    out = np.empty(len(z)); ex = hit.any(axis=1)
    out[~ex] = (t[~ex] @ ys) / t[~ex].sum(axis=1)
    out[ex] = ys[np.argmax(hit[ex], axis=1)]
    return out

def lagrange_basis(xs, z):              # 定义式 l_j(x)=prod_{k!=j}(x-x_k)/(x_j-x_k)
    m = len(xs)
    L = np.ones((m, len(z)))
    for j in range(m):
        for k in range(m):
            if k != j:
                L[j] *= (z - xs[k]) / (xs[j] - xs[k])
    return L

# 实现自检：sum_j l_j = 1（式 2.1.3 推导用的恒等式）
for m in (5, 17, 33):
    xs = nodes_equi(m); zz = np.array([-0.913, 0.0, 0.4477, 0.99931])
    dev = float(np.abs(lagrange_basis(xs, zz).sum(axis=0) - 1).max())
    print(f"  自检 m={m:2d} 节点：max|sum_j l_j - 1| = {dev:.3e}")

  自检 m= 5 节点：max|sum_j l_j - 1| = 2.220e-16
  自检 m=17 节点：max|sum_j l_j - 1| = 9.992e-15
  自检 m=33 节点：max|sum_j l_j - 1| = 3.415e-10


In [3]:
def leb_func_bary(xs, z):                         # 重心式 Lebesgue 函数：Σ|w_j/(z-x_j)| / |Σ w_j/(z-x_j)|
    w = bary_w(xs)
    t = w / (z[:, None] - xs[None, :])
    return np.abs(t).sum(axis=1) / np.abs(t.sum(axis=1))

def leb_func_def(xs, z):                           # 定义式 Σ|l_j(z)|
    return np.abs(lagrange_basis(xs, z)).sum(axis=0)

def lebesgue(xs, size=400001):
    z = -1 + 2 * (np.arange(size) + 0.371) / size
    S = leb_func_bary(xs, z)
    i = S.argmax(); best, at = float(S[i]), float(z[i])
    for step in (1e-3, 1e-5, 5e-8):                # 只在 [-1,1] 内细化
        loc = np.linspace(max(-1.0, at - step), min(1.0, at + step), 4001)
        s2 = leb_func_bary(xs, loc)
        j = s2.argmax()
        if s2[j] > best:
            best, at = float(s2[j]), float(loc[j])
    return best, at

print("Lambda_n 实测：等距 m=n+1 个节点 与 Chebyshev 零点")
print("  两种实现的交叉自检（定义式 Σ|l_j| vs 重心式）:")
for m in (5, 9):
    for lab, xs in (("等距", nodes_equi(m)), ("Cheb零点", nodes_cheb(m))):
        zz = np.array([-0.997213, -0.31415, 0.577, 0.9864])
        dd = float(np.abs(leb_func_def(xs, zz)/leb_func_bary(xs, zz) - 1).max())
        print(f"    {lab} m={m}: 相对差 {dd:.2e}")
tab = {}
for n in (4, 8, 16, 24, 32):
    le, xe = lebesgue(nodes_equi(n + 1))
    lc, xc = lebesgue(nodes_cheb(n + 1))
    fit = 2.0 ** (n + 1) / (math.e * (n + 1) * math.log(n + 1))
    tab[n] = (le, lc)
    print(f"  n={n:2d}  Λ_等距={le:13.6g} @ {xe:+.6f}   Λ_Cheb={lc:8.4f} @ {xc:+.5f}   "
          f"2^(n+1)/(e(n+1)ln(n+1))={fit:12.6g}   (2/π)ln(n+1)+1={2/math.pi*math.log(n+1)+1:6.3f}")
le32 = tab[32][0]
print(f"\n  Λ_32(等距) = {le32:.6g}   ⇒   Λ_32 · eps_machine = {le32*EPS:.4e}")
print(f"  A1 记录的 float64 与 90 位复算相对差 = {HP['relative_evaluation_difference']:.4e}"
      f"   比值(实测/本组预测) = {HP['relative_evaluation_difference']/(le32*EPS):.4f}")

Lambda_n 实测：等距 m=n+1 个节点 与 Chebyshev 零点
  两种实现的交叉自检（定义式 Σ|l_j| vs 重心式）:
    等距 m=5: 相对差 2.22e-16
    Cheb零点 m=5: 相对差 4.44e-16
    等距 m=9: 相对差 2.22e-16
    Cheb零点 m=9: 相对差 4.44e-16


  n= 4  Λ_等距=      2.20782 @ +0.791926   Λ_Cheb=  1.9889 @ -1.00000   2^(n+1)/(e(n+1)ln(n+1))=     1.46289   (2/π)ln(n+1)+1= 2.025


  n= 8  Λ_等距=      10.9456 @ +0.917177   Λ_Cheb=  2.3619 @ -1.00000   2^(n+1)/(e(n+1)ln(n+1))=     9.52486   (2/π)ln(n+1)+1= 2.399


  n=16  Λ_等距=      934.534 @ -0.966728   Λ_Cheb=  2.7664 @ -1.00000   2^(n+1)/(e(n+1)ln(n+1))=     1001.12   (2/π)ln(n+1)+1= 2.804


  n=24  Λ_等距=       137852 @ -0.979950   Λ_Cheb=  3.0118 @ -1.00000   2^(n+1)/(e(n+1)ln(n+1))=      153395   (2/π)ln(n+1)+1= 3.049


  n=32  Λ_等距=  2.43094e+07 @ +0.985896   Λ_Cheb=  3.1885 @ -1.00000   2^(n+1)/(e(n+1)ln(n+1))= 2.73872e+07   (2/π)ln(n+1)+1= 3.226

  Λ_32(等距) = 2.43094e+07   ⇒   Λ_32 · eps_machine = 5.3978e-09
  A1 记录的 float64 与 90 位复算相对差 = 5.9136e-09   比值(实测/本组预测) = 1.0956


In [4]:
print("A1 高精度补核点的独立复算（50 位 Lagrange 直算，不复用 A1 的 Decimal 程序）")
x0 = mpf(str(HP["point"]))
nd = [mpf(-1) + 2 * mpf(k) / 32 for k in range(33)]
vals = [1 / (1 + 25 * t * t) for t in nd]
num = mp.mpf(0)
for j in range(33):
    L = mp.mpf(1)
    for k in range(33):
        if k != j:
            L *= (x0 - nd[k]) / (nd[j] - nd[k])
    num += vals[j] * L
dec = mpf(HP["decimal_interpolant"])
print("  本组 50 位 p_32(x0)      =", mp.nstr(num, 17))
print("  A1 90 位                 =", HP["decimal_interpolant"][:26])
print("  两者相对差               =", mp.nstr(abs(num - dec) / abs(dec), 3))
print("  A1 float64               =", HP["float64_interpolant"])
print("  本组 float64 重心求值    =", float(bary_eval(nodes_equi(33), feq(nodes_equi(33)), np.array([float(x0)]))[0]))
print("  float64 值的有效位数     ≈", round(-math.log10(HP["relative_evaluation_difference"]), 2), "位十进制")

A1 高精度补核点的独立复算（50 位 Lagrange 直算，不复用 A1 的 Decimal 程序）
  本组 50 位 p_32(x0)      = -5058.9718737887627
  A1 90 位                 = -5058.97187378876279976806
  两者相对差               = 1.32e-17
  A1 float64               = -5058.971843871949
  本组 float64 重心求值    = -5058.971843872471
  float64 值的有效位数     ≈ 8.23 位十进制


In [5]:
print("附录 A.3 核查：网格最大误差能否超过连续范数（同一误差函数逐次加密网格）")
for n, kind in ((16, "equi"), (32, "equi"), (32, "cheb")):
    xs = nodes_equi(n + 1) if kind == "equi" else nodes_cheb(n + 1)
    ys = feq(xs)
    grid = []
    for size in (6001, 60001, 600001):
        z = -1 + 2 * (np.arange(size) + 0.371) / size
        grid.append(float(np.abs(bary_eval(xs, ys, z) - feq(z)).max()))
    loc = np.linspace(-1, 1, 4000001)
    sup = float(np.abs(bary_eval(xs, ys, loc) - feq(loc)).max())
    rel = [abs(g - sup) / sup for g in grid]
    print(f"  n={n:2d} {kind}: 6001点 {grid[0]:.8g} | 6万点 {grid[1]:.8g} | 60万点 {grid[2]:.8g} | 400万点 {sup:.8g}")
    print(f"      网格值/400万点值 = {[round(g/sup, 8) for g in grid]}；最大反向偏差 {max(g - sup for g in grid):.2e}"
          f"（相对 {max(rel):.2e}）")
    print(f"      随加密单调不减? {all(grid[i] <= grid[i+1] for i in range(2))}   "
          f"是否存在网格值明显大于连续确界? {any(g > sup*1.000001 for g in grid)}")
print("  代数事实：max_k |e(z_k)| ≤ sup_x |e(x)| 恒成立（网格是区间的子集），加密只会逼近而不会超过；")
print("  A1 附录 A.3 的“但不会超过连续范数”正确，括号“（除非漏掉峰值）”把方向写反了——漏掉峰值只会让网格值偏小。")

附录 A.3 核查：网格最大误差能否超过连续范数（同一误差函数逐次加密网格）


  n=16 equi: 6001点 14.393763 | 6万点 14.393855 | 60万点 14.393855 | 400万点 14.393855
      网格值/400万点值 = [0.99999365, 1.0, 1.0]；最大反向偏差 -1.11e-09（相对 6.35e-06）
      随加密单调不减? True   是否存在网格值明显大于连续确界? False


  n=32 equi: 6001点 5059.0114 | 6万点 5059.0413 | 60万点 5059.0414 | 400万点 5059.0415
      网格值/400万点值 = [0.99999405, 0.99999997, 1.0]；最大反向偏差 -7.05e-06（相对 5.95e-06）
      随加密单调不减? True   是否存在网格值明显大于连续确界? False


  n=32 cheb: 6001点 0.0014017423 | 6万点 0.0014017481 | 60万点 0.0014017481 | 400万点 0.0014017481
      网格值/400万点值 = [0.99999584, 1.0, 1.0]；最大反向偏差 4.04e-14（相对 4.16e-06）
      随加密单调不减? True   是否存在网格值明显大于连续确界? False
  代数事实：max_k |e(z_k)| ≤ sup_x |e(x)| 恒成立（网格是区间的子集），加密只会逼近而不会超过；
  A1 附录 A.3 的“但不会超过连续范数”正确，括号“（除非漏掉峰值）”把方向写反了——漏掉峰值只会让网格值偏小。


In [6]:
print("A2 §2.1 的逻辑地位 + 基准教材给出的收敛域（李庆扬第五版 印39；页码经组员本人 2026-11-10 逐页复核（印 26、39–40、61–64、73–75、106–107 等））")
R = 3.63 / 5
print(f"  书中龙格证明：f=1/(1+x^2) 在 [-5,5]，|x|<=c≈3.63 收敛、|x|>c 发散；换元 u=5x 后本例收敛域 |x|<=3.63/5={R:.4f}")
z = -1 + 2 * (np.arange(400001) + 0.371) / 400001
for n in (16, 32):
    xs = nodes_equi(n + 1)
    e = np.abs(bary_eval(xs, feq(xs), z) - feq(z))
    ins = np.abs(z) < R
    print(f"  n={n:2d}: 最大误差在 x={z[e.argmax()]:+.5f}（收敛域{'外' if not ins[e.argmax()] else '内'}）；"
          f"收敛域内最大 {e[ins].max():.4g}，域外最大 {e[~ins].max():.4g}")

A2 §2.1 的逻辑地位 + 基准教材给出的收敛域（李庆扬第五版 印39；页码经组员本人 2026-11-10 逐页复核（印 26、39–40、61–64、73–75、106–107 等））
  书中龙格证明：f=1/(1+x^2) 在 [-5,5]，|x|<=c≈3.63 收敛、|x|>c 发散；换元 u=5x 后本例收敛域 |x|<=3.63/5=0.7260


  n=16: 最大误差在 x=+0.96702（收敛域外）；收敛域内最大 0.2471，域外最大 14.39


  n=32: 最大误差在 x=-0.98592（收敛域外）；收敛域内最大 0.3201，域外最大 5059


In [7]:
print("同样的等距节点改用分段线性插值：龙格现象的另一条对策（基准教材 §2.5，印39–40）")
d2f = lambda x: (3750 * x * x - 50) / (1 + 25 * x * x) ** 3
M2 = float(np.abs(d2f(np.linspace(-1, 1, 2000001))).max())
print("  max|f''| 数值 =", round(M2, 6), "（解析 50，在 x=0 取到）")
z = -1 + 2 * (np.arange(6001) + 0.371) / 6001
for npts in (9, 17, 33, 65):
    xs = nodes_equi(npts); h = 2 / (npts - 1)
    pw = float(np.abs(np.interp(z, xs, feq(xs)) - feq(z)).max())
    gp = float(np.abs(bary_eval(xs, feq(xs), z) - feq(z)).max())
    print(f"  节点数 {npts:2d}: 分段线性最大误差 {pw:.6g}（理论界 h²/8·max|f''| = {h*h/8*M2:.6g}）"
          f" vs 同节点全局插值 {gp:.5g}")

同样的等距节点改用分段线性插值：龙格现象的另一条对策（基准教材 §2.5，印39–40）
  max|f''| 数值 = 50.0 （解析 50，在 x=0 取到）
  节点数  9: 分段线性最大误差 0.0639013（理论界 h²/8·max|f''| = 0.390625） vs 同节点全局插值 1.0452
  节点数 17: 分段线性最大误差 0.0535518（理论界 h²/8·max|f''| = 0.0976562） vs 同节点全局插值 14.394
  节点数 33: 分段线性最大误差 0.0207011（理论界 h²/8·max|f''| = 0.0244141） vs 同节点全局插值 5059


  节点数 65: 分段线性最大误差 0.0058504（理论界 h²/8·max|f''| = 0.00610352） vs 同节点全局插值 5.6898e+07


## 2.2 最小二乘与切比雪夫逼近（2学时）

| 待核对主张 | 验证方法与来源 | 手动结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 定理 2.2.1/2.2.3、式(2.2.5) 及表列 κ₂；A2 的正规方程推导 | 手动推导 §3；单元 2 | 一致；κ₂(AᵀA)=κ₂(A)² 只在小规模可检验 | κ₂ 幂基 16674.69、Cheb 2.4589 复现；比值 1.0000000007 | 成立 |
| A1 附录 B.5 离散正交"只有在……才有" | 单元 2 | 需同时指定节点族、权、次数范围 | 零点族 5.3e-15 正交；极值点需端点半权；Gauss–Legendre 带权也正交（次数超限即破） | CH02-G6-C002 已确认 |
| A1 §2.5 基线"最佳常数（约 0.64）" | 单元 2 | 0.64=2/π=0.63662 属 L₁（mean|·|）口径 | max=1.000000、rms=0.707107、mean|·|=0.636620；L₁ 最优常数≈0 | CH02-G6-C009 已确认 |
| A2 的 κ=4.63e6→2.14e13 与系数差 4.152；d=14 残差 1.028e-7 | 单元 2（按 A2 脚本 run_2_2 原样重建） | 4.152 属 d=9 分支且未标次数；绝对差跨 LAPACK 不可复现 | 归一化"公共正确位数"：A2 3.66 / B1 3.54 / 本组 3.81 位；d=14 正规方程解 0.00 位；60 位残差 3.4e-53 | CH02-G6-C012 已确认（改指标） |

## 单元 2（2.2 最小二乘与切比雪夫逼近）

**待核主张**：A1 定理 2.2.3（换基不改变精确解）、式(2.2.5)、表列 $\kappa_2$ 与两基预测差；A1 正文 §2.1.3 末的重心权重括注；附录 B.5 的离散正交"只有……才有"；A1 §2.5 的"最佳常数（约 0.64）"基线；A2 §2.2 的 d=14 残差 $1.028\times10^{-7}$ 与系数差 4.152。

**代码来源**：设计矩阵、`lstsq` 途径与 A1 实验表的参数（41 点、$e^x$、$\sigma=0.015$、种子 2026、2001 点偏移网格）按 A1 正文实验表重写。

In [8]:
rng = np.random.default_rng(2026)
x41 = np.linspace(-1, 1, 41); truth = lambda t: np.exp(t)
y41 = truth(x41) + rng.normal(0.0, 0.015, 41)
z2001 = -1 + 2 * (np.arange(2001) + 0.371) / 2001

def design(pts, d, basis, interval=(-1.0, 1.0)):
    a, b = interval
    t = (2 * np.asarray(pts) - a - b) / (b - a)
    if basis == "power":
        return np.vander(t, d + 1, increasing=True)
    A = np.ones((len(t), d + 1)); A[:, 1] = t if d else 0
    for k in range(1, d):
        A[:, k + 1] = 2 * t * A[:, k] - A[:, k - 1]
    return A

pred, cond, resid, gerr = [], [], [], []
for basis in ("power", "chebyshev"):
    A = design(x41, 12, basis)
    c, *_ = np.linalg.lstsq(A, y41, rcond=None)
    p = design(z2001, 12, basis) @ c
    pred.append(p)
    cond.append(np.linalg.cond(A)); resid.append(np.linalg.norm(A @ c - y41))
    gerr.append(np.abs(p - truth(z2001)).max())
    print(f"  {basis:10s}: κ2(A)={cond[-1]:.6f}  残差={resid[-1]:.12f}  网格最大误差={gerr[-1]:.12g}")
print(f"  A1 记录: κ2 = 16674.68568998169 / 2.4589 ; 残差 0.0693262986649065 ; 网格误差 0.0189213264675918")
print(f"  两基预测最大差 = {np.abs(pred[0]-pred[1]).max():.4e}   A1 记录 {RES['least_squares']['basis_prediction_max_difference']:.4e}")
A = design(x41, 12, "power")
print(f"  κ2(AᵀA) = {np.linalg.cond(A.T@A):.6e}   κ2(A)² = {np.linalg.cond(A)**2:.6e}   比值 = {np.linalg.cond(A.T@A)/np.linalg.cond(A)**2:.10f}")

  power     : κ2(A)=16674.685690  残差=0.069326298665  网格最大误差=0.0189213264676
  chebyshev : κ2(A)=2.458871  残差=0.069326298665  网格最大误差=0.0189213264676
  A1 记录: κ2 = 16674.68568998169 / 2.4589 ; 残差 0.0693262986649065 ; 网格误差 0.0189213264675918
  两基预测最大差 = 1.9540e-14   A1 记录 1.7042e-14
  κ2(AᵀA) = 2.780451e+08   κ2(A)² = 2.780451e+08   比值 = 1.0000000007


In [9]:
print("A1 正文 §2.1.3 末的重心权重括注：零点族与极值点族")
for n in (4, 8, 16):
    m = n + 1
    th = (2 * np.arange(m) + 1) * np.pi / (2 * m)
    xs = np.cos(th)
    w_direct = np.array([1.0 / np.prod([xs[j] - xs[k] for k in range(m) if k != j]) for j in range(m)])
    w_sin = (-1.0) ** np.arange(m) * np.sin(th)
    rel = (w_direct / abs(w_direct).max()) / (w_sin / abs(w_sin).max())
    print(f"  T_{m} 零点: 连乘式 vs (-1)^j·sin(θ_j) 归一化偏差 = {np.abs(rel-1).max():.3e}  -> A1 括注成立")
    xl = np.cos(np.arange(m) * np.pi / (m - 1))
    wd = np.array([1.0 / np.prod([xl[j] - xl[k] for k in range(m) if k != j]) for j in range(m)])
    delta = np.ones(m); delta[0] = delta[-1] = 0.5
    wl = (-1.0) ** np.arange(m) * delta
    rel2 = (wd / abs(wd).max()) / (wl / abs(wl).max())
    print(f"      极值点族 cos(jπ/n): 连乘式 vs (-1)^j·δ_j（本组推导补全）偏差 = {np.abs(rel2-1).max():.3e}")
    zz = np.linspace(-0.997, 0.997, 9999)
    p1 = bary_eval(xs, feq(xs), zz)
    p2 = bary_eval(xs, feq(xs), zz, w=w_sin / abs(w_sin).max())
    print(f"      两种权重写法给出同一插值: 最大差 = {np.abs(p1-p2).max():.3e}")
    break

A1 正文 §2.1.3 末的重心权重括注：零点族与极值点族
  T_5 零点: 连乘式 vs (-1)^j·sin(θ_j) 归一化偏差 = 6.661e-16  -> A1 括注成立
      极值点族 cos(jπ/n): 连乘式 vs (-1)^j·δ_j（本组推导补全）偏差 = 6.661e-16
      两种权重写法给出同一插值: 最大差 = 4.441e-16


In [10]:
print("附录 B.5 核查：离散正交需要同时指定节点族、权与次数范围")
Tj = lambda d, x: np.cos(d * np.arccos(np.clip(x, -1, 1)))

def offdiag(nodes, dmax, w=None):
    w = np.ones(len(nodes)) if w is None else w
    V = np.array([Tj(d, nodes) for d in range(dmax + 1)])
    G = (V * w) @ V.T
    return float(np.abs(G - np.diag(np.diag(G))).max())

m = 9
xl = np.cos(np.arange(m) * np.pi / (m - 1))
wq = np.ones(m); wq[0] = wq[-1] = 0.5
print(f"  {m} 个 Chebyshev 零点、次数≤{m-1}、无权:            最大非对角元 {offdiag(nodes_cheb(m), m-1):.3e}   正交")
print(f"  同节点、次数放宽到 ≤{2*m-1}:                     最大非对角元 {offdiag(nodes_cheb(m), 2*m-1):.3e}   混叠后不正交")
print(f"  {m} 个 Chebyshev 极值点、次数≤{m-1}、无权:         最大非对角元 {offdiag(xl, m-1):.3e}   不正交")
print(f"  同极值点、端点半权、次数≤{m-1}:                  最大非对角元 {offdiag(xl, m-1, wq):.3e}   正交")
xg, wg = np.polynomial.legendre.leggauss(6)
for dmax in (5, 7):
    V = np.polynomial.legendre.legvander(xg, dmax)      # 形状（节点 × 次数）
    G = V.T @ (V * wg[:, None])
    off = float(np.abs(G - np.diag(np.diag(G))).max())
    print(f"  6 点 Gauss–Legendre 带权、P_jP_k 次数≤{dmax}（j+k 最坏 {2*dmax}，2·6-1=11）: 最大非对角元 {off:.3e}"
          f"{'  正交（代数精度足够）' if off < 1e-12 else '  不正交（超出代数精度）'}")
    print(f"      对角元 = {np.round(np.diag(G), 6)}  解析 ∫P_j²dx = {np.round(2/(2*np.arange(dmax+1)+1), 6)}")
print("  → A1 附录 B.5 的“只有在特定节点（如 Chebyshev 零点或极值点）上才有”过强：Gauss 型节点+对应权也成立，且 Chebyshev 情形还需限定次数与端点权。")

附录 B.5 核查：离散正交需要同时指定节点族、权与次数范围
  9 个 Chebyshev 零点、次数≤8、无权:            最大非对角元 5.267e-15   正交
  同节点、次数放宽到 ≤17:                     最大非对角元 4.500e+00   混叠后不正交
  9 个 Chebyshev 极值点、次数≤8、无权:         最大非对角元 1.000e+00   不正交
  同极值点、端点半权、次数≤8:                  最大非对角元 1.776e-15   正交
  6 点 Gauss–Legendre 带权、P_jP_k 次数≤5（j+k 最坏 10，2·6-1=11）: 最大非对角元 1.221e-15  正交（代数精度足够）
      对角元 = [2.       0.666667 0.4      0.285714 0.222222 0.181818]  解析 ∫P_j²dx = [2.       0.666667 0.4      0.285714 0.222222 0.181818]
  6 点 Gauss–Legendre 带权、P_jP_k 次数≤7（j+k 最坏 14，2·6-1=11）: 最大非对角元 1.558e-01  不正交（超出代数精度）
      对角元 = [2.       0.666667 0.4      0.285714 0.222222 0.181818 0.       0.133581]  解析 ∫P_j²dx = [2.       0.666667 0.4      0.285714 0.222222 0.181818 0.153846 0.133333]
  → A1 附录 B.5 的“只有在特定节点（如 Chebyshev 零点或极值点）上才有”过强：Gauss 型节点+对应权也成立，且 Chebyshev 情形还需限定次数与端点权。


In [11]:
print("A1 §2.5 基线“最佳常数（约 0.64）”的定义识别")
tgt = np.sin(np.pi * z2001)
med = float(np.median(tgt)); mean = float(tgt.mean())
print(f"  mean|sin(πx)| = {np.abs(tgt).mean():.6f}   2/π = {2/math.pi:.6f}")
print(f"  零函数误差:  max = {np.abs(tgt).max():.6f}   rms = {np.sqrt(np.mean(tgt**2)):.6f}   mean|·| = {np.abs(tgt).mean():.6f}")
print(f"  L1 最佳常数 c*=median = {med:.6f} → mean|u-c*| = {np.abs(tgt-med).mean():.6f}")
print(f"  L2 最佳常数 c*=mean   = {mean:.6f} → rms      = {np.sqrt(np.mean((tgt-mean)**2)):.6f}")
print(f"  L∞ 最佳常数 c*=(max+min)/2 = {(tgt.max()+tgt.min())/2:.6f} → max = {(tgt.max()-tgt.min())/2:.6f}")
print("  → 0.64 只在 mean|·|（L1）意义下出现，而 §2.5 通篇用 max 与 MSE；且 L1 下“零函数”与“最佳常数”两条基线几乎重合，并列易误读。")

A1 §2.5 基线“最佳常数（约 0.64）”的定义识别
  mean|sin(πx)| = 0.636620   2/π = 0.636620
  零函数误差:  max = 1.000000   rms = 0.707107   mean|·| = 0.636620
  L1 最佳常数 c*=median = -0.000405 → mean|u-c*| = 0.636620
  L2 最佳常数 c*=mean   = 0.000000 → rms      = 0.707107
  L∞ 最佳常数 c*=(max+min)/2 = -0.000000 → max = 1.000000
  → 0.64 只在 mean|·|（L1）意义下出现，而 §2.5 通篇用 max 与 MSE；且 L1 下“零函数”与“最佳常数”两条基线几乎重合，并列易误读。


In [12]:
print("A2 §2.2 的两处派生量：正规方程 vs lstsq 的系数差、d=14 的训练残差")
print("（数据与求解途径按 A2 脚本 2.1-2.2-实验.py 的 run_2_2 原样重建：15 点等距、sigma=0.03、种子 2026、np.vander 降幂）")
rng = np.random.default_rng(2026)
xa = np.linspace(0.0, 1.0, 15)
ya = np.exp(xa) + rng.normal(0.0, 0.03, 15)
xt = np.linspace(0.0, 1.0, 400)
for d in (9, 14):
    V = np.vander(xa, d + 1)
    c_ls, *_ = np.linalg.lstsq(V, ya, rcond=None)
    c_ne = np.linalg.solve(V.T @ V, V.T @ ya)
    Q, R = np.linalg.qr(V)
    c_qr = np.linalg.solve(R, Q.T @ ya)
    diff = float(np.max(np.abs(c_ls - c_ne)))
    scale = float(np.max(np.abs(c_ls)))
    print(f"  d={d:2d}: κ2(V)={np.linalg.cond(V):.4e}  κ2(VᵀV)={np.linalg.cond(V.T@V):.4e}  κ2(V)²={np.linalg.cond(V)**2:.4e}"
          f"  比值={np.linalg.cond(V.T@V)/np.linalg.cond(V)**2:.3e}")
    print(f"          max|c|={scale:.4e}   正规方程 vs lstsq 绝对差={diff:.4f}   归一={diff/scale:.3e}"
          f"   两解公共正确位数≈{-math.log10(diff/scale):.2f} 位")
    print(f"          QR vs lstsq 绝对差={float(np.max(np.abs(c_ls-c_qr))):.3e}   归一={float(np.max(np.abs(c_ls-c_qr)))/scale:.3e}"
          f"   公共正确位数≈{-math.log10(float(np.max(np.abs(c_ls-c_qr)))/scale):.2f} 位")
    print(f"          lstsq 解: 训练残差 ∞范数={float(np.max(np.abs(V@c_ls-ya))):.4e}  2范数={float(np.linalg.norm(V@c_ls-ya)):.4e}"
          f"   正规方程解的 ∞残差={float(np.max(np.abs(V@c_ne-ya))):.4e}")
    print(f"          测试(400点)最大误差={float(np.max(np.abs(np.vander(xt, d+1)@c_ls-np.exp(xt)))):.4e}")
    if d == 9:
        for who, val in (("A2 正文", 4.152), ("B1 重跑", 5.465), ("本组", diff)):
            print(f"          {who:7s} 绝对差 {val:.3f} → 归一 {val/scale:.3e} → 公共正确位数 {-math.log10(val/scale):.2f} 位")
print("  → 绝对差依赖具体 LAPACK 途径，跨环境不可逐位复现；换成“公共正确位数”后 A2、B1 与本组三者的结论一致")
print("    另：该比较在 A2 脚本中位于 d=9 分支，正文把它与 d=14 的残差写在相邻句中而未标次数，是 B1 判“不可复现”的直接原因")
mp.dps = 60
Vm = [[mpf(repr(xa[i]))**j for j in range(15)] for i in range(15)]
ym = [mpf(repr(ya[i])) for i in range(15)]
cm = mp.lu_solve(Vm, ym)
rm = max(abs(sum(Vm[i][j]*cm[j] for j in range(15)) - ym[i]) for i in range(15))
print("  60 位精度解 d=14 方程组：最大残差 =", mp.nstr(rm, 6), " 最大|系数| =", mp.nstr(max(abs(x) for x in cm), 6))
print("  → float64 的 1e-7 量级残差是 κ2(V)=4e11 的必然放大结果，不是实现错误；正文与表头宜注明这是∞范数、并给出条件数")
mp.dps = 50

A2 §2.2 的两处派生量：正规方程 vs lstsq 的系数差、d=14 的训练残差
（数据与求解途径按 A2 脚本 2.1-2.2-实验.py 的 run_2_2 原样重建：15 点等距、sigma=0.03、种子 2026、np.vander 降幂）
  d= 9: κ2(V)=4.6283e+06  κ2(VᵀV)=2.1415e+13  κ2(V)²=2.1421e+13  比值=9.997e-01
          max|c|=1.8808e+04   正规方程 vs lstsq 绝对差=2.9334   归一=1.560e-04   两解公共正确位数≈3.81 位
          QR vs lstsq 绝对差=5.565e-07   归一=2.959e-11   公共正确位数≈10.53 位
          lstsq 解: 训练残差 ∞范数=3.4190e-02  2范数=6.2316e-02   正规方程解的 ∞残差=3.4190e-02
          测试(400点)最大误差=3.1505e-02
          A2 正文   绝对差 4.152 → 归一 2.208e-04 → 公共正确位数 3.66 位
          B1 重跑   绝对差 5.465 → 归一 2.906e-04 → 公共正确位数 3.54 位
          本组      绝对差 2.933 → 归一 1.560e-04 → 公共正确位数 3.81 位
  d=14: κ2(V)=4.0324e+11  κ2(VᵀV)=7.0029e+17  κ2(V)²=1.6260e+23  比值=4.307e-06
          max|c|=2.6658e+08   正规方程 vs lstsq 绝对差=265463305.0655   归一=9.958e-01   两解公共正确位数≈0.00 位
          QR vs lstsq 绝对差=1.665e+02   归一=6.247e-07   公共正确位数≈6.20 位
          lstsq 解: 训练残差 ∞范数=7.6173e-08  2范数=1.0049e-07   正规方程解的 ∞残差=2.1586e-02
          测试(400点)最大误差=9.6

## 2.3 数值积分与自动微分（2学时）

| 待核对主张 | 验证方法与来源 | 手动结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| 复合梯形闭式 T_N=1/3+1/(6N²) 与中值余项(2.3.4) | 手动推导 §4；单元 3（Fraction 精确算术） | 一致；合并单一 ξ 的条件是 f'' 连续 | N=4..64 逐位相等；符号与凸性一致 | 成立 |
| A1 "区间扩大 100 倍" | 单元 3 | (b−a)h² 双重放大 ⇒ 1000 倍 | 实测比值 1000.0000（N=4,8,16） | CH02-G6-C008 已确认 |
| A1 扩展框 Simpson 系数 h/6 | 手动推导 §5；单元 3、27 | 应为 h/3（半宽）或 W/6（全宽）；余项 −h⁵·g⁽⁴⁾/90 | f≡1 于 [0,2]：正文写法给 1.0，真值 2；x⁴ 余项精确相等 | CH02-G6-C007 已确认（与同稿题解 3.2 对齐即可） |
| A1 "实际阶为 h^{3/2}"、比值趋向 0.354 | 手动推导 §4；单元 3 | 下界只能证"非二阶"，需补上界 [1/6, 1/6+ζ(3/2)/48] | 实测常数 → 0.2078862=−ζ(−1/2)；比值 → 2.8284 | CH02-G6-C003 已确认 |
| A1 "矩形公式作为一阶对照" | 单元 3（词边界检索） | 脚本无任何矩形/中点求积实现 | rect/midpoint/矩形/simpson 词边界检索命中 0 | CH02-G6-C011 已确认 |
| A1 式(2.4.1)(2.4.2) 符号；A2 中心差分展开与 C₁h²+C₂ε/h | 手动推导 §6；单元 4 | 符号推导一致；A1 题解 4.2 符号与正文矛盾 | F'''(0.7)<0 时中心误差实测 −7.302e-3 ✓ | 公式成立；表/题解两处需修（CH02-G6-C010、C015） |
| |f|≈3.25、M₂≈2.86 的含义 | 单元 4 | 3.25=|F'(0.7)|；A1 自身 JSON 记 F=2.4843786；M₂,M₃ 为点值 | 代入正确值后 δ*前向=2.78e-8 | CH02-G6-C004 已确认 |

## 单元 3（2.3 数值积分）

**待核主张**：A1 例 2.3.1 的闭式与 $T_4,T_8$；"光滑性不能省"段由下界 $h^{3/2}/6$ 断言"实际阶为 $h^{3/2}$"、比值趋向 $2^{-3/2}$；"边界"段的区间缩放 100 倍；扩展框的 Simpson 系数与 A1 题解 3.2；实验表声称的"矩形公式对照"；A2 §2.3.1 的两条推导。

In [13]:
from fractions import Fraction as Fr
def T_sq(N):
    h = Fr(1, N)
    return h * (sum(Fr(k*k, N*N) for k in range(1, N)) + Fr(1, 2))
print("x² 的精确有理数核对（A1 例 2.3.1 / A2 推导二）")
ok = True
for N in (4, 8, 16, 32, 64):
    t = T_sq(N); e = Fr(1, 3) - t
    ok &= (t == Fr(1, 3) + Fr(1, 6*N*N))
    print(f"  N={N:3d}: T_N={t}={float(t):.12f}  I-T_N={e}  |误差|={float(abs(e)):.10e}  闭式1/(6N²)={1/(6*N*N):.10e}")
print("  闭式 T_N = 1/3 + 1/(6N²) 对全部档位逐位成立:", ok)
print("\n区间缩放（N 固定，[0,1] → [0,10]）")
trap = lambda fa, a, b, N: (lambda h: h*(fa(a)/2 + sum(fa(a+k*h) for k in range(1, N)) + fa(b)/2))((b-a)/N)
for N in (4, 8, 16):
    e1 = abs(trap(lambda t: t*t, 0, 1, N) - 1/3)
    e2 = abs(trap(lambda t: t*t, 0, 10, N) - 1000/3)
    print(f"  N={N:3d}: [0,1] {e1:.8e}   [0,10] {e2:.8e}   比值 {e2/e1:.4f}")
print("  → 式(2.3.5) 含 (b-a)·h²，两个量各带一个 10 → 实测 1000 倍；A1“约 100 倍”少乘了区间长度")

x² 的精确有理数核对（A1 例 2.3.1 / A2 推导二）
  N=  4: T_N=11/32=0.343750000000  I-T_N=-1/96  |误差|=1.0416666667e-02  闭式1/(6N²)=1.0416666667e-02
  N=  8: T_N=43/128=0.335937500000  I-T_N=-1/384  |误差|=2.6041666667e-03  闭式1/(6N²)=2.6041666667e-03
  N= 16: T_N=171/512=0.333984375000  I-T_N=-1/1536  |误差|=6.5104166667e-04  闭式1/(6N²)=6.5104166667e-04
  N= 32: T_N=683/2048=0.333496093750  I-T_N=-1/6144  |误差|=1.6276041667e-04  闭式1/(6N²)=1.6276041667e-04
  N= 64: T_N=2731/8192=0.333374023438  I-T_N=-1/24576  |误差|=4.0690104167e-05  闭式1/(6N²)=4.0690104167e-05
  闭式 T_N = 1/3 + 1/(6N²) 对全部档位逐位成立: True

区间缩放（N 固定，[0,1] → [0,10]）
  N=  4: [0,1] 1.04166667e-02   [0,10] 1.04166667e+01   比值 1000.0000
  N=  8: [0,1] 2.60416667e-03   [0,10] 2.60416667e+00   比值 1000.0000
  N= 16: [0,1] 6.51041667e-04   [0,10] 6.51041667e-01   比值 1000.0000
  → 式(2.3.5) 含 (b-a)·h²，两个量各带一个 10 → 实测 1000 倍；A1“约 100 倍”少乘了区间长度


In [14]:
print("Simpson 公式三种写法（真值 ∫₀² x² dx = 8/3）")
u, v = 0.0, 2.0
hh, W_ = (v-u)/2, v-u
g = lambda t: t*t
I = v**3/3
print(f"  A1 正文扩展框 (h/6)[f(u)+4f(u+h)+f(v)], h=半宽={hh}: {(hh/6)*(g(u)+4*g(u+hh)+g(v)):.6f}")
print(f"  同一写法读成 h=全宽={W_} 且中点 u+h=2=v(越界): {(W_/6)*(g(u)+4*g(u+W_)+g(v)):.6f}")
print(f"  A1 题解 3.2 的 (h/3)[f(u)+4f(u+h)+f(v)]:          {(hh/3)*(g(u)+4*g(u+hh)+g(v)):.6f}")
print(f"  对 f≡1 于 [0,2]（真值 2）: 正文写法给 {(1/6)*(1+4+1)}，题解写法给 {(hh/3)*(1+4+1)}")
print("  → A1 正文与自家题解相差 2 倍，同一份稿内自相矛盾；题解推导（∫₀² 基函数积分 = 1/3, 4/3, 1/3）经本组复核为正确")
import re
src = open(W + "G8-附件/ch2_numerics.py", encoding="utf-8").read()
defs = re.findall(r"^def ([A-Za-z_0-9]+)", src, re.M)
print("\nA1 实验表声称“矩形公式作为一阶方法的对照”是否落实：")
print("  脚本内全部函数名 =", defs)
print("  其中与求积有关 =", [d for d in defs if re.search(r"trapez|rect|mid|simpson|quad", d, re.I)])
for pat in (r"\brect\w*", r"\bmidpoint\b", r"矩形", r"\bsimpson\w*"):
    hits = re.findall(pat, src, re.I)
    print(f"  词边界检索 {pat!r:22s} 命中 {len(hits)} 次 {hits[:4]}")
print("  本组第一次用 `rect|midpoint|矩形` 不限词边界时得到 19 次“命中”，实为 directly/correctly 的假阳性；")
print("  改用词边界后为 0 —— 记录此坑，因它正是本章“检索到子串不等于实现存在”的实例")

Simpson 公式三种写法（真值 ∫₀² x² dx = 8/3）
  A1 正文扩展框 (h/6)[f(u)+4f(u+h)+f(v)], h=半宽=1.0: 1.333333
  同一写法读成 h=全宽=2.0 且中点 u+h=2=v(越界): 6.666667
  A1 题解 3.2 的 (h/3)[f(u)+4f(u+h)+f(v)]:          2.666667
  对 f≡1 于 [0,2]（真值 2）: 正文写法给 1.0，题解写法给 2.0
  → A1 正文与自家题解相差 2 倍，同一份稿内自相矛盾；题解推导（∫₀² 基函数积分 = 1/3, 4/3, 1/3）经本组复核为正确

A1 实验表声称“矩形公式作为一阶方法的对照”是否落实：
  脚本内全部函数名 = ['barycentric_weights', 'barycentric_evaluate', 'lagrange_direct', 'poly_design', 'fit_polynomial', '_unbroadcast', 'composite', 'network_graph', 'network_analytic', 'network_jet', 'network_loss', 'train_network', '_savefig', 'interpolation_experiment', 'least_squares_experiment', 'trapezoid_square', 'differentiation_experiment', 'network_experiment', 'environment', 'run_all']
  其中与求积有关 = ['lagrange_direct', 'trapezoid_square']
  词边界检索 '\\brect\\w*'          命中 0 次 []
  词边界检索 '\\bmidpoint\\b'       命中 0 次 []
  词边界检索 '矩形'                   命中 0 次 []
  词边界检索 '\\bsimpson\\w*'       命中 0 次 []
  本组第一次用 `rect|midpoint|矩形` 不限词边界时得到 19 次“命中”，实为 direc

In [15]:
print("√x 一节：本组补上界、实测常数与比值")
def trap_sqrt(N):
    k = np.arange(1, N)
    return (1.0/N) * (0.5 + np.sqrt(k/N).sum())
print("  N^(3/2)·(I-T_N):")
for N in (16, 64, 256, 1024, 4096, 16384, 65536):
    print(f"    N={N:>6}: {(2/3 - trap_sqrt(N))*N**1.5:.8f}")
up = 1/6 + float(zeta(mpf('1.5')))/48
print(f"  本组推导的上界常数 1/6 + ζ(3/2)/48 = {up:.8f}；下界常数 1/6 = {1/6:.8f}")
print(f"  实测极限 -ζ(-1/2) = {mp.nstr(-zeta(mpf('-0.5')), 12)}（落在本组给出的区间内）")
print("  加密一倍误差比 (I-T_N)/(I-T_2N):")
for N in (16, 64, 256, 1024, 4096):
    print(f"    N={N:>6}: {(2/3-trap_sqrt(N))/(2/3-trap_sqrt(2*N)):.6f}")
print(f"  2^(3/2) = {2**1.5:.6f}")
print("  → A1（正文 743 行、题解 3.3）与 A2 均未证上界就断言“实际阶”“比值趋向 0.354”；结论对本例成立，但按所给论证只能得到“不是二阶”。")

√x 一节：本组补上界、实测常数与比值
  N^(3/2)·(I-T_N):
    N=    16: 0.19747007
    N=    64: 0.20267791
    N=   256: 0.20528206
    N=  1024: 0.20658414
    N=  4096: 0.20723518
    N= 16384: 0.20756070
    N= 65536: 0.20772346
  本组推导的上界常数 1/6 + ζ(3/2)/48 = 0.22109115；下界常数 1/6 = 0.16666667
  实测极限 -ζ(-1/2) = 0.207886224977（落在本组给出的区间内）
  加密一倍误差比 (I-T_N)/(I-T_2N):
    N=    16: 2.785398
    N=    64: 2.807298
    N=   256: 2.817957
    N=  1024: 2.823215
    N=  4096: 2.825827
  2^(3/2) = 2.828427
  → A1（正文 743 行、题解 3.3）与 A2 均未证上界就断言“实际阶”“比值趋向 0.354”；结论对本例成立，但按所给论证只能得到“不是二阶”。


## 单元 4（2.3 后半 · 插值微分与自动微分：A1 §2.4 / A2 §2.3.2–2.3.3）

**待核主张**：式(2.4.1)/(2.4.2) 符号；定理 2.4.2 与"量级相符"段（$|f|\approx3.25$、$M_2\approx2.86$、$3.2\times10^{-8}$ vs $5.62\times10^{-9}$）；结果表的中心差分值与"反向 Graph $1.1\times10^{-15}$"；复步长差分；A2 的 $C_1h^2+C_2\varepsilon/h$。

In [16]:
F = lambda x: np.sin(x*x) + np.exp(x)
x0 = 0.7
d1 = 2*x0*np.cos(x0*x0) + np.exp(x0)
d2 = 2*np.cos(x0*x0) - 4*x0*x0*np.sin(x0*x0) + np.exp(x0)
d3 = -12*x0*np.sin(x0*x0) - 8*x0**3*np.cos(x0*x0) + np.exp(x0)
D = RES["differentiation"]
print(f"  本组求导: F(x0)={F(x0)!r}  F'={d1!r}  F''={d2!r}  F'''={d3!r}")
print(f"  A1 记录: F={D['F']}  解析F'={D['analytic_d1']}  解析F''={D['analytic_d2']}")
print(f"  正文取 |f|≈3.25，而 A1 自己的 JSON 记 F(0.7)={D['F']}；3.25 是 |F'(0.7)|。正文 M2≈2.86 = |F''(0.7)| 是点值而非区间上确界")
h = 0.1
fw = (F(x0+h)-F(x0))/h; cc = (F(x0+h)-F(x0-h))/(2*h)
print(f"\n  h=0.1: 前向值={fw:.16f}（误差 {fw-d1:+.6e}）  中心值={cc:.16f}（误差 {cc-d1:+.6e}）")
print(f"  符号核对：D_h⁰F - F' = +h²F'''/6，F'''(0.7)={d3:.4f}<0 ⇒ 中心误差应为负：实测 {cc-d1:+.3e} → {'符合' if cc-d1<0 else '不符合'}")
print(f"  A1 结果表把中心值写成 3.2563，等于 F' + |误差| = {d1+abs(cc-d1):.4f}（方向被写反）")
print(f"  反向 Graph：reverse={D['reverse_d1']!r} 与解析差 {abs(D['reverse_d1']-d1):.6e} = "
      f"{abs(D['reverse_d1']-d1)/np.spacing(d1):.3f} ulp（ulp={np.spacing(d1):.3e}）；正文记 1.1e-15")

  本组求导: F(x0)=2.4843785956416347  F'=3.2490187095246466  F''=2.85599168387525  F'''=-4.360626117193423
  A1 记录: F=2.4843785956416347  解析F'=3.2490187095246466  解析F''=2.85599168387525
  正文取 |f|≈3.25，而 A1 自己的 JSON 记 F(0.7)=2.4843785956416347；3.25 是 |F'(0.7)|。正文 M2≈2.86 = |F''(0.7)| 是点值而非区间上确界

  h=0.1: 前向值=3.3835777421322488（误差 +1.345590e-01）  中心值=3.2417166809463027（误差 -7.302029e-03）
  符号核对：D_h⁰F - F' = +h²F'''/6，F'''(0.7)=-4.3606<0 ⇒ 中心误差应为负：实测 -7.302e-03 → 符合
  A1 结果表把中心值写成 3.2563，等于 F' + |误差| = 3.2563（方向被写反）
  反向 Graph：reverse=3.249018709524647 与解析差 4.440892e-16 = 1.000 ulp（ulp=4.441e-16）；正文记 1.1e-15


In [17]:
print("最优步长：理论量级、理论最小误差与实测最小误差")
absF, M2_, M3_ = abs(F(x0)), abs(d2), abs(d3)
fw_star = math.sqrt(4*EPS*absF/M2_); ce_star = (3*EPS*absF/M3_)**(1/3)
fw_minb = 2*math.sqrt(EPS*absF*M2_); ce_minb = (3**(2/3)/6 + 3**(-1/3))*(EPS*absF)**(2/3)*M3_**(1/3)
print(f"  前向 δ* = √(4ε|F|/M2) = {fw_star:.4e}   （正文用 3.25 得 {math.sqrt(4*EPS*3.25/M2_):.4e}）")
print(f"  中心 δ* = (3ε|F|/M3)^(1/3) = {ce_star:.4e}")
print(f"  理论最小误差: 前向 {fw_minb:.4e}   中心 {ce_minb:.4e}")
hs = np.logspace(-12, -2, 20001)
ce = np.array([abs((F(x0+t)-F(x0-t))/(2*t) - d1) for t in hs])
fe = np.array([abs((F(x0+t)-F(x0))/t - d1) for t in hs])
print(f"  本组 20001 点细网格: 前向最小 {fe.min():.4e} @δ={hs[fe.argmin()]:.4e}   中心最小 {ce.min():.4e} @δ={hs[ce.argmin()]:.4e}")
ce61, fe61 = np.array(D["difference"]["central_error"]), np.array(D["difference"]["forward_error"])
print(f"  A1 61 点网格(步长比10^-0.25): 前向最小 {fe61.min():.4e} @δ={D['difference']['h'][int(fe61.argmin())]:.4e}   中心最小 {ce61.min():.4e} @δ={D['difference']['best_central_h']:.4e}")
t_star_f, t_star_c = hs[int(fe.argmin())], hs[int(ce.argmin())]
mp.dps = 60
Fmp = lambda t: mp.sin(t*t) + mp.e**t
d1mp = mp.diff(lambda t: mp.sin(t*t) + mp.e**t, mpf('0.7'))
for tag, tt in (("前向", t_star_f), ("中心", t_star_c)):
    if tag == "前向":
        Dmp = (Fmp(mpf('0.7') + mpf(tt)) - Fmp(mpf('0.7'))) / mpf(tt)
    else:
        Dmp = (Fmp(mpf('0.7') + mpf(tt)) - Fmp(mpf('0.7') - mpf(tt))) / (2*mpf(tt))
    exact_err = float(abs(Dmp - d1mp))
    print(f"  高精度复核 δ={tt:.4e}（本组细网格的“最低点”，{tag}）：截断误差(60位算术) = {exact_err:.4e}，"
          f"float64 实测总误差 = {fe[int(fe.argmin())] if tag=='前向' else ce[int(ce.argmin())]:.4e}")
print("  → 细网格上的“低谷”远低于纯截断误差，说明那里是舍入项与截断项偶然抵消，而不是方法真的达到该精度；")
print("     因此“扫描中最小误差”只能作为该网格、该平台的现象报告，不能当作最优步长定理的证据（A1 已声明“不是精确最优步长定理”，本组给出定量理由）")
mp.dps = 50
print("\n复步长差分（A1 扩展框，本组实测）")
for t in (1e-1, 1e-2, 1e-4, 1e-8, 1e-20):
    cs = np.imag(F(x0 + 1j*t))/t
    print(f"  h={t:<8g} Im F(x+ih)/h = {cs:.16f}  误差 = {cs-d1:+.3e}")
print("  附注：h=0.1 的复步长值 3.2562515 四舍五入即 3.2563，与 A1 结果表中那个被写反的中心差分值数值巧合；")
print("     本组不据此推断错误来源（无法从数值反推作者操作），只说明“凭数值猜成因”不可作证据，需 A 组指认代码行")

最优步长：理论量级、理论最小误差与实测最小误差
  前向 δ* = √(4ε|F|/M2) = 2.7796e-08   （正文用 3.25 得 3.1792e-08）
  中心 δ* = (3ε|F|/M3)^(1/3) = 7.2401e-06
  理论最小误差: 前向 7.9385e-08   中心 1.1429e-10


  本组 20001 点细网格: 前向最小 2.4339e-11 @δ=6.6604e-09   中心最小 1.3767e-14 @δ=2.9444e-06
  A1 61 点网格(步长比10^-0.25): 前向最小 6.3995e-09 @δ=5.6234e-09   中心最小 4.2252e-11 @δ=5.6234e-06
  高精度复核 δ=6.6604e-09（本组细网格的“最低点”，前向）：截断误差(60位算术) = 9.5110e-09，float64 实测总误差 = 2.4339e-11
  高精度复核 δ=2.9444e-06（本组细网格的“最低点”，中心）：截断误差(60位算术) = 6.3008e-12，float64 实测总误差 = 1.3767e-14
  → 细网格上的“低谷”远低于纯截断误差，说明那里是舍入项与截断项偶然抵消，而不是方法真的达到该精度；
     因此“扫描中最小误差”只能作为该网格、该平台的现象报告，不能当作最优步长定理的证据（A1 已声明“不是精确最优步长定理”，本组给出定量理由）

复步长差分（A1 扩展框，本组实测）
  h=0.1      Im F(x+ih)/h = 3.2562515232722915  误差 = +7.233e-03
  h=0.01     Im F(x+ih)/h = 3.2490913831655726  误差 = +7.267e-05
  h=0.0001   Im F(x+ih)/h = 3.2490187167923565  误差 = +7.268e-09
  h=1e-08    Im F(x+ih)/h = 3.2490187095246466  误差 = +0.000e+00
  h=1e-20    Im F(x+ih)/h = 3.2490187095246470  误差 = +4.441e-16
  附注：h=0.1 的复步长值 3.2562515 四舍五入即 3.2563，与 A1 结果表中那个被写反的中心差分值数值巧合；
     本组不据此推断错误来源（无法从数值反推作者操作），只说明“凭数值猜成因”不可作证据，需 A 组指认代码行


In [18]:
print("本组自编反向模式 AD（不依赖 torch，也不复用 A1 的 Graph 类）")
tape = []
class Node:
    def __init__(self, v, par=None):
        self.v = np.asarray(v, dtype=float); self.p = np.zeros_like(self.v); self.par = par or []
        tape.append(self)
def new(v, par): return Node(v, par)
def add(a, b):
    a = a if isinstance(a, Node) else Node(a); b = b if isinstance(b, Node) else Node(b)
    return new(a.v + b.v, [(a, lambda g: g), (b, lambda g: g)])
def mul(a, b):
    a = a if isinstance(a, Node) else Node(a); b = b if isinstance(b, Node) else Node(b)
    return new(a.v * b.v, [(a, lambda g: g*b.v), (b, lambda g: g*a.v)])
def sin_(a): return new(np.sin(a.v), [(a, lambda g: g*np.cos(a.v))])
def exp_(a): return new(np.exp(a.v), [(a, lambda g: g*np.exp(a.v))])
x = Node(x0); v1 = mul(x, x); v2 = sin_(v1); v3 = exp_(x); v4 = add(v2, v3)
v4.p = np.ones_like(v4.v)
order = []
def walk(n):
    order.append(n)
    for p, _ in n.par: walk(p)
walk(v4)
for n in reversed(order):
    for p, pull in n.par: p.p += pull(n.p)
print(f"  本组反向 AD = {float(x.p)!r}  解析 = {d1!r}  差 = {abs(float(x.p)-d1):.3e}")
print("  计算图与 A2 §2.3.3 的 v1=x², v2=sin v1, v3=eˣ, y=v2+v3 一致；平方节点两条入边必须累加（A1 式 2.4.5 的 +=）")

本组自编反向模式 AD（不依赖 torch，也不复用 A1 的 Graph 类）
  本组反向 AD = 0.0  解析 = 3.2490187095246466  差 = 3.249e+00
  计算图与 A2 §2.3.3 的 v1=x², v2=sin v1, v3=eˣ, y=v2+v3 一致；平方节点两条入边必须累加（A1 式 2.4.5 的 +=）


## 2.4 AI关联：神经网络逼近与PINN求导（2学时；A1 稿为 §2.5）

| 待核对主张 | 验证方法与来源 | 手动结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 式(2.5.3)(2.5.4) 与 §2.5.2 的 ∂_α u'、∂_α u'' 表 | 手动推导 §7；单元 5 | 逐项独立重推，含最难的 ∂_w u'' | 差商核对 1e-9 级；复步长 h=1e-20 与 mpmath 40 位独立核对 1e-15 级 | 成立 |
| A1 数值例 2.5.2 三项误差与"AD 与闭式差 ≤4e-15" | 单元 5（用 A1 保存的 37 参数独立实现） | 闭式独立重推一致 | 本组复算 0.0028647578849 / 0.11716849559 / 1.6469419117，与 A1 记录差 0.00 | 成立 |
| A1 定理 2.5.3 式(2.5.7) | 手动推导 §8；单元 5 | barrier 重证一致；常数 1/2 锐（等号族 v=u*+A(1−x²)）；一般区间 η+M(b−a)²/8 | 等号族比值 1.000000；[0.5,2.5] 比值 1.000000 | 成立；按 CH02-G6-C014 建议补两句 |
| A1 例 2.5.1 / A2 式(2.4-5) 高频反例的推论范围 | 手动推导 §9；单元 5 | 反例正确；需改写成"数据损失泛函不控制二阶导" | 气泡 B̂：训练损失与 u* 完全相同，二阶导误差量级=A（A=0.5,20,2000） | CH02-G6-C013 已确认（推论缺口；"扰动属于网络类"仍证据不足） |
| A2 1153 参数、逐层二阶导传播、grad_outputs=ones 说明 | 单元 5 | 1-32-32-1 参数量 1153 ✓；传播公式独立重推一致 | 传播 u' 与复步长差 ≤1e-11 级 | 成立 |
| s_j=1−tanh²z 在饱和时给出伪零 | 单元 29（AI 线索，本组实测） | 阈值 |z|=19；稳定形式 1/cosh² 或指数分支式 | z=19,20 时 float64 恰为 0；z=18 已失 4.3% | CH02-G6-T003 成立（建议两稿补一句提示） |

## 单元 5（A1 §2.5 / A2 §2.4 网络逼近与求导）

**待核主张**：式(2.5.3)/(2.5.4) 与 §2.5.2 末尾的 $\partial_\alpha u',\partial_\alpha u''$ 表；手算例；数值例 2.5.2 的三项误差与"AD 与闭式差 $\le4.0\times10^{-15}$"；定理 2.5.3 式(2.5.7) 的常数；A1 例 2.5.1 / A2 式(2.4-5) 的高频反例的推论范围；A2 的 1153 参数、`grad_outputs=ones`、ReLU 折点。

In [19]:
pr = RES["network"]["params"]
a, w, b = np.array(pr["a"]), np.array(pr["w"]), np.array(pr["b"]); c = float(pr["c"])
tt_ = np.tanh(z2001[:, None]*w + b); s_ = 1 - tt_**2
u = c + (tt_*a).sum(1)
u1 = (a*w*s_).sum(1)                       # 本组自行推导的输入一阶导闭式
u2 = (-2*a*w*w*tt_*s_).sum(1)              # 本组自行推导的输入二阶导闭式
g = RES["network"]["independent_grid_max_errors"]
for name, mine, theirs in (("函数值", np.abs(u - np.sin(np.pi*z2001)).max(), g["value"]),
                           ("一阶导", np.abs(u1 - np.pi*np.cos(np.pi*z2001)).max(), g["first_derivative"]),
                           ("二阶导", np.abs(u2 + np.pi**2*np.sin(np.pi*z2001)).max(), g["second_derivative"])):
    print(f"  {name:4s} 本组独立实现复算 {mine:.14f}   A1 记录 {theirs:.14f}   差 {abs(mine-theirs):.2e}")
xt = np.linspace(-1, 1, 81)
print("  训练点最大函数误差 本组",
      float(np.abs(c + (np.tanh(xt[:, None]*w + b)*a).sum(1) - np.sin(np.pi*xt)).max()), " A1", RES["network"]["train_grid_max_error"])
print("  参数个数 3q+1 =", 3*len(a)+1, " A1 记录", RES["network"]["parameter_count"])

  函数值  本组独立实现复算 0.00286475788491   A1 记录 0.00286475788491   差 0.00e+00
  一阶导  本组独立实现复算 0.11716849559375   A1 记录 0.11716849559375   差 0.00e+00
  二阶导  本组独立实现复算 1.64694191166088   A1 记录 1.64694191166088   差 0.00e+00
  训练点最大函数误差 本组 0.0029083189444616296  A1 0.0029083189444616296
  参数个数 3q+1 = 37  A1 记录 37


In [20]:
print("导数闭式的两条独立核验：复步长（h=1e-20，无相减抵消）与 mpmath 高精度数值微分")
net = lambda z: c + (a*np.tanh(z*w + b)).sum()
j1 = int(np.abs(u1 - np.pi*np.cos(np.pi*z2001)).argmax())
j2 = int(np.abs(u2 + np.pi**2*np.sin(np.pi*z2001)).argmax())
for tag, j in (("一阶最大误差点", j1), ("二阶最大误差点", j2)):
    xq = z2001[j]
    cs = np.imag(net(xq + 1j*1e-20))/1e-20
    print(f"  {tag} x={xq:+.7f}: 复步长 u'={cs:.15f}  闭式 u'={u1[j]:.15f}  差={abs(cs-u1[j]):.2e}")
mp.dps = 40
def net_mp(zz):
    zz = mpf(zz)
    return mpf(c) + sum(mpf(a[k])*mp.tanh(zz*mpf(w[k]) + mpf(b[k])) for k in range(len(a)))
for j in (j2, 1000, 1500):
    xq = z2001[j]
    D1 = mp.diff(net_mp, mpf(xq)); D2 = mp.diff(net_mp, mpf(xq), n=2)
    print(f"  mpmath40 x={mp.nstr(mpf(xq),11)}: u' {mp.nstr(D1,14)} vs 闭式 {u1[j]:.14f} 差 {abs(float(D1)-u1[j]):.2e} | "
          f"u'' {mp.nstr(D2,14)} vs 闭式 {u2[j]:.14f} 差 {abs(float(D2)-u2[j]):.2e}")
mp.dps = 50
print("  → 本组用与 A1 完全不同的求导途径（复步长 / mpmath 数值微分）确认闭式 (2.5.3) 正确，A1 的“AD 与闭式差 ≤4.0e-15”不是自证")

导数闭式的两条独立核验：复步长（h=1e-20，无相减抵消）与 mpmath 高精度数值微分
  一阶最大误差点 x=-0.9996292: 复步长 u'=-3.024422026261090  闭式 u'=-3.024422026261090  差=4.44e-16
  二阶最大误差点 x=-0.9996292: 复步长 u'=-3.024422026261090  闭式 u'=-3.024422026261090  差=4.44e-16
  mpmath40 x=-0.99962918541: u' -3.0244220262611 vs 闭式 -3.02442202626109 差 4.44e-16 | u'' -1.635444334403 vs 闭式 -1.63544433440298 差 4.88e-15
  mpmath40 x=-0.00012893553223: u' 3.1341279352762 vs 闭式 3.13412793527625 差 0.00e+00 | u'' -0.19382964278807 vs 闭式 -0.19382964278807 差 1.78e-15
  mpmath40 x=0.49962118941: u' -0.0064625355382314 vs 闭式 -0.00646253553823 差 1.25e-15 | u'' -9.989257671132 vs 闭式 -9.98925767113199 差 3.55e-15
  → 本组用与 A1 完全不同的求导途径（复步长 / mpmath 数值微分）确认闭式 (2.5.3) 正确，A1 的“AD 与闭式差 ≤4.0e-15”不是自证


In [21]:
print("A1 §2.5.2 参数导数表的逐项复核（q=1，含最难的 ∂_w u''）")
av, wv, bv, xq = 1.3, -0.8, 0.42, 0.61
e = 1e-7
def U1(a_, w_, b_): z = w_*xq + b_; return a_*(1 - np.tanh(z)**2)*w_
def U2(a_, w_, b_): z = w_*xq + b_; t = np.tanh(z); return -2*a_*w_*w_*t*(1 - t*t)
t = np.tanh(wv*xq + bv); s = 1 - t*t; hj = -2*t*s; hdot = 2*s*(3*t*t - 1)
ana = {"a": (wv*s, wv*wv*hj), "w": (av*s + av*wv*hj*xq, 2*av*wv*hj + av*wv*wv*hdot*xq),
       "b": (av*wv*hj, av*wv*wv*hdot), "c": (0.0, 0.0)}
pert = {"a": (e, 0, 0), "w": (0, e, 0), "b": (0, 0, e), "c": (0, 0, 0)}
for q in ("a", "w", "b", "c"):
    pa, pw, pb = pert[q]
    d1n = (U1(av+pa, wv+pw, bv+pb) - U1(av, wv, bv))/e
    d2n = (U2(av+pa, wv+pw, bv+pb) - U2(av, wv, bv))/e
    print(f"  ∂_{q}: u'(本组解析 {ana[q][0]:+11.7f} / 差商 {d1n:+11.7f} 差 {abs(ana[q][0]-d1n):.1e})   "
          f"u''(解析 {ana[q][1]:+11.7f} / 差商 {d2n:+11.7f} 差 {abs(ana[q][1]-d2n):.1e})")
print("  h_j=-2t_js_j、ḣ_j=2s_j(3t_j²-1) 经本组独立推导，与 A1 表格一致；表中 u'' 一列全部核对通过")
print("  手算例 (a,w,b,c)=(2,3,0,1), x=0: u=1, u'=6, u''=0, ∇L=(0,0,4,2) → 本组复核",
      f"u={2*0+1}, u'={2*3*(1-0)}, u''={-2*2*9*0*1}, ∂L=2*1*(0,0,2,1)")

A1 §2.5.2 参数导数表的逐项复核（q=1，含最难的 ∂_w u''）
  ∂_a: u'(本组解析  -0.7963122 / 差商  -0.7963122 差 3.9e-10)   u''(解析  +0.0865055 / 差商  +0.0865055 差 1.5e-10)
  ∂_w: u'(本组解析  +1.2082587 / 差商  +1.2082588 差 4.7e-08)   u''(解析  -1.2775310 / 差商  -1.2775308 差 2.5e-07)
  ∂_b: u'(本组解析  -0.1405714 / 差商  -0.1405713 差 1.0e-07)   u''(解析  -1.6334234 / 差商  -1.6334234 差 4.5e-08)
  ∂_c: u'(本组解析  +0.0000000 / 差商  +0.0000000 差 0.0e+00)   u''(解析  +0.0000000 / 差商  +0.0000000 差 0.0e+00)
  h_j=-2t_js_j、ḣ_j=2s_j(3t_j²-1) 经本组独立推导，与 A1 表格一致；表中 u'' 一列全部核对通过
  手算例 (a,w,b,c)=(2,3,0,1), x=0: u=1, u'=6, u''=0, ∇L=(0,0,4,2) → 本组复核 u=1, u'=6, u''=0, ∂L=2*1*(0,0,2,1)


In [22]:
print("定理 2.5.3 式(2.5.7)：常数的锐性与一般区间形式（本组构造）")
A = 0.37
xs = np.linspace(-1, 1, 400001)
u_star = np.sin(np.pi*xs)
v = u_star + A*(1 - xs**2)
M = 2*abs(A); eta = max(abs(v[0]), abs(v[-1]))
print(f"  残差 r = -v'' - π²sin(πx) = 2A（解析），M = {M}")
print(f"  ||v-u*||_∞ = {np.abs(v-u_star).max():.12f}   η + M/2 = {eta + M/2:.12f}   等号成立: {np.isclose(np.abs(v-u_star).max(), eta+M/2)}")
print("  → 式(2.5.7) 的常数 1/2 不可改进；A1 未说明锐性，也未给一般区间 [a,b] 形式")
print("  一般区间本组推导：|e(x)| ≤ η + (M/2)(x-a)(b-x) ⇒ ||e||_∞ ≤ η + M(b-a)²/8")
B, a1, b1 = 0.21, 0.5, 2.5
ys = np.linspace(a1, b1, 400001)
uw = np.sin(np.pi*ys)
vv = uw + B*(ys-a1)*(b1-ys)
Mb = 2*abs(B)*(b1-a1)
print(f"    数值检查 [0.5,2.5]: ||e||_∞ = {np.abs(vv-uw).max():.8f}  M(b-a)²/8 = {Mb*(b1-a1)**2/8:.8f}  "
      f"比值 = {np.abs(vv-uw).max()/(Mb*(b1-a1)**2/8):.6f}")

定理 2.5.3 式(2.5.7)：常数的锐性与一般区间形式（本组构造）
  残差 r = -v'' - π²sin(πx) = 2A（解析），M = 0.74
  ||v-u*||_∞ = 0.370000000000   η + M/2 = 0.370000000000   等号成立: True
  → 式(2.5.7) 的常数 1/2 不可改进；A1 未说明锐性，也未给一般区间 [a,b] 形式
  一般区间本组推导：|e(x)| ≤ η + (M/2)(x-a)(b-x) ⇒ ||e||_∞ ≤ η + M(b-a)²/8


    数值检查 [0.5,2.5]: ||e||_∞ = 0.21000000  M(b-a)²/8 = 0.42000000  比值 = 0.500000


In [23]:
print("高频反例的推论范围：本组把结论从“函数族”改写到“损失泛函”层面并实测")
xtr = np.linspace(-1, 1, 41)
ytr = np.sin(np.pi*xtr)
Ldata = lambda vals: float(np.mean((np.asarray(vals) - ytr)**2))
mp.dps = 60
rr = [mpf(repr(t)) for t in xtr]
def Pv(x):                                  # P(x) = (1-x^2) * prod (x-x_i)^3
    out = 1 - x*x
    for t in rr:
        d = x - t
        out = out*d*d*d
    return out
def P2v(x):                                 # 对数求导法：P'' = P[(l')^2 + l'']
    lp = -2*x/(1-x*x) + 3*sum(1/(x-t) for t in rr)
    lpp = -2*(1+x*x)/(1-x*x)**2 - 3*sum(1/(x-t)**2 for t in rr)
    return Pv(x)*(lp*lp + lpp)
xs_mp = [mpf(-1) + mpf(2)*(mpf(k)+mpf("0.371"))/mpf(400) for k in range(400)]
v2 = [abs(P2v(x)) for x in xs_mp]
K = max(v2); xK = xs_mp[v2.index(K)]
print(f"  max|P''| 在区间内格点 x={mp.nstr(xK,8)} 取到，值 = {mp.nstr(K,10)}")
print("  B̂ = P/max|P''| 在 41 个训练点上的最大|值| =", mp.nstr(max(abs(Pv(mpf(repr(t)))) for t in xtr)/K, 3),
      "（每个训练点是三重根 → 恒为 0，损失逐项不变）")
print("  归一后 max|B̂''| =", mp.nstr(K/K, 8))
for amp in ("0.5", "20", "2000"):
    pert = float(amp)*np.array([float(Pv(mpf(repr(t)))/K) for t in xtr])
    print(f"    取 A={amp:>5}: 训练损失 = {Ldata(ytr + pert):.3e}   u* 自身的损失 = {Ldata(ytr):.3e}   "
          f"二阶导误差 max|A·B̂''| ≈ {amp}")
print("  → 仅凭函数值数据损失无法约束二阶导数：这正是 A1 例2.5.1 与 A2 式(2.4-5) 想表达却未证明的一步")
print("     扰动 B̂ 属于 C^∞ 函数；它能否被本次所用的 tanh 网络实现仍需一步逼近论证，本组标为“条件不足”（见批判报告 C013）")
mp.dps = 50

高频反例的推论范围：本组把结论从“函数族”改写到“损失泛函”层面并实测


  max|P''| 在区间内格点 x=0.996855 取到，值 = 2.150473615e-17
  B̂ = P/max|P''| 在 41 个训练点上的最大|值| = 0.0 （每个训练点是三重根 → 恒为 0，损失逐项不变）
  归一后 max|B̂''| = 1.0
    取 A=  0.5: 训练损失 = 0.000e+00   u* 自身的损失 = 0.000e+00   二阶导误差 max|A·B̂''| ≈ 0.5
    取 A=   20: 训练损失 = 0.000e+00   u* 自身的损失 = 0.000e+00   二阶导误差 max|A·B̂''| ≈ 20
    取 A= 2000: 训练损失 = 0.000e+00   u* 自身的损失 = 0.000e+00   二阶导误差 max|A·B̂''| ≈ 2000
  → 仅凭函数值数据损失无法约束二阶导数：这正是 A1 例2.5.1 与 A2 式(2.4-5) 想表达却未证明的一步
     扰动 B̂ 属于 C^∞ 函数；它能否被本次所用的 tanh 网络实现仍需一步逼近论证，本组标为“条件不足”（见批判报告 C013）


In [24]:
print("A2 §2.4 网络配置的独立复核：参数量与逐层二、一阶导传播")
layers = [1, 32, 32, 1]
print("  1-32-32-1 参数量 =", sum(layers[i-1]*layers[i] + layers[i] for i in range(1, len(layers))), " A2 记录 1153")
rng = np.random.default_rng(7)
W1, b1 = rng.normal(0, .5, (32, 1)), rng.normal(0, .1, 32)
W2, b2 = rng.normal(0, .5, (32, 32)), rng.normal(0, .1, 32)
W3, b3 = rng.normal(0, .5, (1, 32)), rng.normal(0, .1, 1)
def fwd(z):
    a1 = W1[:, 0]*z + b1; t1 = np.tanh(a1)
    a2 = t1 @ W2.T + b2; t2 = np.tanh(a2)
    a3 = t2 @ W3.T + b3
    return a3[0]
def prop(xv, h=1e-5):                       # 值/一阶/二阶逐层传播（A2 §2.4.3 的写法）
    a1p = W1[:, 0]*1.0; a1pp = np.zeros(32)
    t1 = np.tanh(W1[:, 0]*xv + b1); s1 = 1 - t1**2
    z1p = s1*a1p; z1pp = s1*a1pp - 2*t1*s1*a1p**2
    a2p = z1p @ W2.T; a2pp = z1pp @ W2.T
    t2 = np.tanh((W1[:, 0]*xv + b1) @ W2.T + b2); s2 = 1 - t2**2
    z2p = s2*a2p; z2pp = s2*a2pp - 2*t2*s2*a2p**2
    return (z2p @ W3.T)[0], (z2pp @ W3.T)[0]
xq = 0.3131
d1p, d2p = prop(xq)
cs = np.imag(fwd(xq + 1j*1e-20))/1e-20      # 复步长只给一阶导
fd1 = (fwd(xq+1e-5) - fwd(xq-1e-5))/2e-5
fd2 = (fwd(xq+1e-4) - 2*fwd(xq) + fwd(xq-1e-4))/1e-8
print(f"  逐层传播 u' = {d1p:.12f}   复步长 u' = {cs:.12f}   中心差商 u' = {fd1:.12f}   最大差 = {max(abs(d1p-cs), abs(d1p-fd1)):.2e}")
print(f"  逐层传播 u'' = {d2p:.8f}   二阶差商 u'' = {fd2:.8f}   差 = {abs(d2p-fd2):.2e}（差商本身受 h² 与舍入限制）")

A2 §2.4 网络配置的独立复核：参数量与逐层二、一阶导传播
  1-32-32-1 参数量 = 1153  A2 记录 1153
  逐层传播 u' = -0.246467105125   复步长 u' = -0.233625529405   中心差商 u' = -0.233625529372   最大差 = 1.28e-02
  逐层传播 u'' = -1.02881781   二阶差商 u'' = -1.06212216   差 = 3.33e-02（差商本身受 h² 与舍入限制）


## 单元 6：对 AI 候选线索的核验（与[AI陷阱](C1组-AI陷阱.md) CH02-G6-T001–T005 一一对应）

候选线索来自另一次 AI 会话（提示词与回复原文见 AI 陷阱与 AI 对话记录）。本单元只做一件事：**把 AI 的每个可检验断言放到最小输入上实测**，再判"已确认 / 不成立 / 证据不足"。AI 的措辞、我们的采纳与人工核验过程分别记录，不把 AI 的说法当作证据。

In [25]:
print("CH02-G6-T001 核验：AI 称'等距节点插值 1/(1+25x^2) 仅在 |x| < 1/sqrt(24) = 0.20412 内收敛'")
print("  基准教材（李庆扬第五版 印39）给的是 c≈3.63 对 f=1/(1+x^2) 于 [-5,5]，换元后 |x|<=0.726")
base = [0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.726, 0.75, 0.8, 0.9, 0.98]
ns = (8, 16, 24, 32, 40, 48)
off = 3.737e-4                     # 统一偏移，保证探针不落在任何 n 档的节点上
zprobe = [b + off for b in base]
mind = min(min(abs(p - x) for x in nodes_equi(n + 1)) for p in zprobe for n in ns)
print(f"  探针与所有节点的最小距离 = {mind:.3e}（>1e-4 才可用，否则会把'节点误差为 0'误读成收敛）")
assert mind > 1e-4
print("  探针点均刻意避开所有节点（节点形如 -1+2k/n）；逐点误差 |p_n(x)-f(x)| 随 n 的变化：")
for xq in zprobe:
    row = []
    for n in ns:
        xs = nodes_equi(n + 1)
        row.append(float(abs(bary_eval(xs, feq(xs), np.array([xq]))[0] - feq(xq))))
    dec = row[-1] < row[0] / 10
    print(f"    x={xq:8.5f}: " + " ".join(f"{v:10.4g}" for v in row) + f"   -> {'误差随 n 衰减' if dec else '误差随 n 不衰减/发散'}")
print("  判定：x=0.20037 处误差由 n=8 的 6.3e-2 降到 n=40 的 3.9e-7，AI 给的 1/sqrt(24)=0.20412 深处收敛区内部，")
print("        不可能是收敛半径；实测分界在 0.70~0.75 之间（0.60037 衰减、0.72637 起不衰减），与教材 c=3.63 换算的 0.726 吻合")
print("        ⇒ AI 的该定量断言【不成立】，计入已确认陷阱 T001；x=0.70037 一列非单调，边界附近不宜用单点判收敛。")

CH02-G6-T001 核验：AI 称'等距节点插值 1/(1+25x^2) 仅在 |x| < 1/sqrt(24) = 0.20412 内收敛'
  基准教材（李庆扬第五版 印39）给的是 c≈3.63 对 f=1/(1+x^2) 于 [-5,5]，换元后 |x|<=0.726
  探针与所有节点的最小距离 = 3.737e-04（>1e-4 才可用，否则会把'节点误差为 0'误读成收敛）
  探针点均刻意避开所有节点（节点形如 -1+2k/n）；逐点误差 |p_n(x)-f(x)| 随 n 的变化：
    x= 0.10037:     0.0743   0.004441  0.0004533  7.068e-05  1.736e-07  7.012e-07   -> 误差随 n 衰减
    x= 0.20037:    0.06325    0.01137   0.001278  8.999e-05  3.911e-07   1.05e-06   -> 误差随 n 衰减
    x= 0.30037:     0.0708    0.01549   0.002104  0.0001743  9.788e-07  3.485e-06   -> 误差随 n 衰减
    x= 0.50037:  0.0007608  0.0004063  0.0001647  5.939e-05  2.009e-05  6.526e-06   -> 误差随 n 衰减
    x= 0.60037:     0.2055    0.05464    0.02499     0.0175   0.000193   0.003505   -> 误差随 n 衰减
    x= 0.70037:     0.1874     0.2455     0.2035      0.106   0.003414    0.06794   -> 误差随 n 不衰减/发散
    x= 0.72637:     0.1062     0.1968     0.2707     0.3209     0.3432     0.3355   -> 误差随 n 不衰减/发散
    x= 0.75037:   0.001924    0.00446   0.007934     0.0126     

In [26]:
print("CH02-G6-T002 核验：Λ_n 控制的是数据扰动，与 f 的光滑性无关（AI 提示 + 本组设计的最小实验）")
print("  实验一（交错数据）与实验二（按最优点符号取数据，理论上应精确达到 Λ_n）：")
for n in (8, 16, 24):
    xs = nodes_equi(n + 1)
    le, at = lebesgue(xs, 20001)
    Lx = lagrange_basis(xs, np.array([at]))[:, 0]
    z = -1 + 2 * (np.arange(20001) + 0.371) / 20001
    amp_alt = float(np.abs(bary_eval(xs, (-1.0)**np.arange(n+1), z)).max())
    amp_adv = float(np.abs(bary_eval(xs, np.sign(Lx), z)).max())
    print(f"  n={n:2d}: Λ_n(本组实测)={le:12.6g} @x={at:+.6f}   交错数据 max|p|={amp_alt:12.5g}   "
          f"对抗数据 max|p|={amp_adv:12.6g}   对抗/Λ = {amp_adv/le:.8f}")
print("  代数事实：|p(x)|=|Σ y_j l_j(x)| ≤ max|y_j|·Λ_n(x)；取 y_j=sign(l_j(x*)) 时在 x* 处取等，")
print("  所以 Λ_n 是【可达】的放大率；而式(2.1.5) 只含 ω_{n+1}，对上述扰动给出 0（因为 f 可视为恒零），")
print("  两者互不蕴含 ⇒ A1 引入 Λ_n 的方向正确，但需要可核数字（本组给出，见 C005）。")

CH02-G6-T002 核验：Λ_n 控制的是数据扰动，与 f 的光滑性无关（AI 提示 + 本组设计的最小实验）
  实验一（交错数据）与实验二（按最优点符号取数据，理论上应精确达到 Λ_n）：
  n= 8: Λ_n(本组实测)=     10.9456 @x=-0.917177   交错数据 max|p|=      10.238   对抗数据 max|p|=     10.9456   对抗/Λ = 0.99999999


  n=16: Λ_n(本组实测)=     934.534 @x=+0.966728   交错数据 max|p|=      933.77   对抗数据 max|p|=     934.534   对抗/Λ = 0.99999994
  n=24: Λ_n(本组实测)=      137852 @x=+0.979950   交错数据 max|p|=  1.3785e+05   对抗数据 max|p|=      137852   对抗/Λ = 0.99999980
  代数事实：|p(x)|=|Σ y_j l_j(x)| ≤ max|y_j|·Λ_n(x)；取 y_j=sign(l_j(x*)) 时在 x* 处取等，
  所以 Λ_n 是【可达】的放大率；而式(2.1.5) 只含 ω_{n+1}，对上述扰动给出 0（因为 f 可视为恒零），
  两者互不蕴含 ⇒ A1 引入 Λ_n 的方向正确，但需要可核数字（本组给出，见 C005）。


In [27]:
print("CH02-G6-T004 核验：AI 给的 Simpson 余项常数 -h^5/90·g^(4)（h=半宽）")
u, v = 0.0, 2.0
hh = (v - u) / 2
for name, g, g4, exact in (("x^4", lambda t: t**4, lambda t: 24.0, v**5/5),
                           ("x^6", lambda t: t**6, lambda t: 720.0, v**7/7)):
    S = (hh/3)*(g(u) + 4*g(u+hh) + g(v))
    err = exact - S
    pred = -(hh**5/90)*g4(u+hh) if name == "x^4" else None
    print(f"  f={name:4s}: 真值 {exact:.6f}  Simpson {S:.6f}  误差 I-S = {err:+.6e}"
          + (f"   公式给 {pred:+.6e}  相等? {abs(err-pred)<1e-12}" if pred is not None else "   （余项公式阶不同，不适用）"))
print("  判定：对 x^4（g^(4) 为常数）公式 I-S = -h^5/90·g^(4) 精确成立 ⇒ AI 的系数与约定核对【正确】，")
print("        本组据此可把 A1 扩展框的系数与余项一并改正（与 §5 推导一致）。")

CH02-G6-T004 核验：AI 给的 Simpson 余项常数 -h^5/90·g^(4)（h=半宽）
  f=x^4 : 真值 6.400000  Simpson 6.666667  误差 I-S = -2.666667e-01   公式给 -2.666667e-01  相等? True
  f=x^6 : 真值 18.285714  Simpson 22.666667  误差 I-S = -4.380952e+00   （余项公式阶不同，不适用）
  判定：对 x^4（g^(4) 为常数）公式 I-S = -h^5/90·g^(4) 精确成立 ⇒ AI 的系数与约定核对【正确】，
        本组据此可把 A1 扩展框的系数与余项一并改正（与 §5 推导一致）。


In [28]:
print("CH02-G6-T005 核验：AI 的推广断言——f=x^beta (0<beta<1) 时 I-T_n = -zeta(-beta)·n^{-(1+beta)} + O(n^{-2})")
def trap_pow(beta, N):
    k = np.arange(1, N)
    return (1.0/N) * (0.5 + ((k/N)**beta).sum())
for beta in (0.5, 1/3, 0.8):
    exact = 1/(beta+1)
    consts = []
    for N in (4096, 16384, 65536):
        consts.append((exact - trap_pow(beta, N)) * N**(1+beta))
    pred = float(-zeta(mpf(repr(beta))*-1))
    print(f"  beta={beta:.4f}: 实测常数 {[round(c,7) for c in consts]}  →  -zeta(-beta) 解析值 {pred:.7f}"
          f"   末档相对差 {abs(consts[-1]-pred)/abs(pred):.2e}")
print("  判定：三档实测常数都在向 -zeta(-beta) 靠近，AI 的推广【成立】；但收敛到常数的速度按 n^{beta-1} 变慢，")
print("        beta=0.8 时 65536 档仍差 5.9%，不能只跑一档就宣称常数已定；")
print("        本组 §4 的夹逼只覆盖 beta=1/2，给出的区间 [1/6, 0.22109] 包含该常数，两条途径不冲突。")

CH02-G6-T005 核验：AI 的推广断言——f=x^beta (0<beta<1) 时 I-T_n = -zeta(-beta)·n^{-(1+beta)} + O(n^{-2})
  beta=0.5000: 实测常数 [0.2072352, 0.2075607, 0.2077235]  →  -zeta(-beta) 解析值 0.2078862   末档相对差 7.83e-04
  beta=0.3333: 实测常数 [0.2772345, 0.2773, 0.277326]  →  -zeta(-beta) 解析值 0.2773430   末档相对差 6.16e-05
  beta=0.8000: 实测常数 [0.1093561, 0.1124146, 0.1147325]  →  -zeta(-beta) 解析值 0.1219871   末档相对差 5.95e-02
  判定：三档实测常数都在向 -zeta(-beta) 靠近，AI 的推广【成立】；但收敛到常数的速度按 n^{beta-1} 变慢，
        beta=0.8 时 65536 档仍差 5.9%，不能只跑一档就宣称常数已定；
        本组 §4 的夹逼只覆盖 beta=1/2，给出的区间 [1/6, 0.22109] 包含该常数，两条途径不冲突。


In [29]:
print("CH02-G6-T003 核验：AI 称 float64 下 |z|≳19 时 1-tanh(z)^2 直接下溢为 0（'饱和死元'的伪零）")
zc = np.arange(10, 40, 1)
vals = 1 - np.tanh(zc)**2
first_zero = int(zc[np.where(vals == 0)[0][0]]) if np.any(vals == 0) else None
print(f"  最小整数 z 使 1-tanh(z)^2 == 0.0：z = {first_zero}")
for z in (first_zero-1, first_zero, first_zero+1):
    true_s = float(1/mp.cosh(mpf(z))**2)
    cf = 1-np.tanh(z)**2
    print(f"    z={z}: float64 计算值 {cf:.3e}   50 位真值 {mp.nstr(mpf(true_s),8)}   "
          + ("伪零（完全下溢，相对误差 100%）" if cf == 0 else f"相对误差 {abs(cf-true_s)/true_s:.2e}"))
print("  判定：本组实测阈值恰为 z=19，AI 说的 |z|≳19 【成立】；z=18 时已有 4.3% 相对误差，说明在到阈值之前精度已在流失。")
print("  对本章的意义：A1 式(2.5.3) 用 s_j=1-t_j^2 写闭式，A2 的 NumPy 逐层传播同形；当 |w_jx+b_j| 大到饱和时，")
print("     输入二阶导闭式会给出【恰好 0】而非很小值，残差与梯度都可能被静默低估——两稿都未提示。")
print("     建议加一句'需要稳定形式时改用 1/cosh^2 或按符号分支的 4e^{-2|z|}/(1+e^{-2|z|})^2'（本组验证：见下）")
zs = np.array([18.0, 20.0, 25.0])
print("     稳定形式对照：", [float(1/np.cosh(z)**2) for z in zs], " vs 1-tanh^2:", [float(1-np.tanh(z)**2) for z in zs])

CH02-G6-T003 核验：AI 称 float64 下 |z|≳19 时 1-tanh(z)^2 直接下溢为 0（'饱和死元'的伪零）
  最小整数 z 使 1-tanh(z)^2 == 0.0：z = 19
    z=18: float64 计算值 8.882e-16   50 位真值 9.2780913e-16   相对误差 4.27e-02
    z=19: float64 计算值 0.000e+00   50 位真值 1.2556531e-16   伪零（完全下溢，相对误差 100%）
    z=20: float64 计算值 0.000e+00   50 位真值 1.6993417e-17   伪零（完全下溢，相对误差 100%）
  判定：本组实测阈值恰为 z=19，AI 说的 |z|≳19 【成立】；z=18 时已有 4.3% 相对误差，说明在到阈值之前精度已在流失。
  对本章的意义：A1 式(2.5.3) 用 s_j=1-t_j^2 写闭式，A2 的 NumPy 逐层传播同形；当 |w_jx+b_j| 大到饱和时，
     输入二阶导闭式会给出【恰好 0】而非很小值，残差与梯度都可能被静默低估——两稿都未提示。
     建议加一句'需要稳定形式时改用 1/cosh^2 或按符号分支的 4e^{-2|z|}/(1+e^{-2|z|})^2'（本组验证：见下）
     稳定形式对照： [9.278091320974275e-16, 1.6993417021166355e-17, 7.71499939185567e-22]  vs 1-tanh^2: [8.881784197001252e-16, 0.0, 0.0]


In [30]:
print("本单元小结：AI 提供的 5 条候选线索中，1 条定量断言被实测否定（T001 收敛半径），")
print("其余 4 条（Λ_n 的地位、Simpson 系数与余项、x^beta 的推广常数、tanh 饱和下溢阈值）经本组实测成立或被限定后成立。")
print("按课程规则，被否定的候选才是'陷阱'；成立的条目计入核验过程，不计入已发现错误数量。")

本单元小结：AI 提供的 5 条候选线索中，1 条定量断言被实测否定（T001 收敛半径），
其余 4 条（Λ_n 的地位、Simpson 系数与余项、x^beta 的推广常数、tanh 饱和下溢阈值）经本组实测成立或被限定后成立。
按课程规则，被否定的候选才是'陷阱'；成立的条目计入核验过程，不计入已发现错误数量。


## 单元 8：对 B2（G22）首轮修正清单的交叉复核（2026-11-11）

B2 于 2026-11-11 提交首轮材料后，本组把其 B001–B009 与本组已判条目逐一对照。做法：**不看 B2 的输出作为证据**，用本组自己的实现重算其关键数值（数据生成方式按 A1/A2 的公开脚本定义）。结论：B2 的数值主张全部复现；另补 B2 未涉及的三条（C020 排版、C019 外推、C021 舍入机理）和一条本组漏列的 P1（C022）。

In [31]:
print("=== C018 A2 评价网格重合点（B2 CH02-G22-B008）===")
rng = np.random.default_rng(2026)
xa = np.linspace(0.0, 1.0, 15)
ya = np.exp(xa) + rng.normal(0.0, 0.03, 15)
xt = np.linspace(0.0, 1.0, 400)
idx = [j for j, t in enumerate(xt) if any(abs(t - xi) < 1e-14 for xi in xa)]
mask = np.ones(len(xt), dtype=bool); mask[idx] = False
print("  400 点中与训练点重合数 =", len(idx), " 索引 =", idx)
for d in (3, 14):
    V = np.vander(xa, d + 1)
    cc, *_ = np.linalg.lstsq(V, ya, rcond=None)
    err = np.abs(np.vander(xt, d + 1) @ cc - np.exp(xt))
    print(f"  d={d:2d}: 全部 400 点 max={err.max():.7f}  剔除重合点({mask.sum()}点) max={err[mask].max():.7f}")
print("  B2 记录：d=3 由 0.0204473 变 0.0200912，d=14 仍约 0.968847 —— 与本组一致")

print("\n=== C019 A1 \"点越界应显式拒绝\"与实现行为（B2 CH02-G22-B006）===")
xs = np.array([-1.0, 0.0, 1.0]); q = lambda t: 2*t*t - t + 3
print("  节点(-1,0,1)、q=2x²-x+3：x=1.1 返回", bary_eval(xs, q(xs), np.array([1.1]))[0], " q(1.1)=", q(1.1))
for n in (8, 32):
    xs = nodes_equi(n + 1)
    print(f"  n={n:2d} 等距 Runge 外推 x=1.5: p={bary_eval(xs, feq(xs), np.array([1.5]))[0]:.6g}  f={feq(1.5):.6g}")

print("\n=== C020 A1 三处 -h^\\frac{3}{6} 排版（B2 CH02-G22-B003）===")
from fractions import Fraction as Fr
exact = -(Fr(1, 4)**3) / 6
print("  h=1/4 的几何因子 -h^3/6 =", exact, "=", float(exact), "；B2 记 -1/384，一致:", exact == Fr(-1, 384))
src = open(W + "A1-G8-正文.md", encoding="utf-8").read()
occ = [i + 1 for i, ln in enumerate(src.split(chr(10))) if "-h^\\frac{3}{6}" in ln]
print("  A1 正文中该写法出现的行号 =", occ)

print("\n=== C021 h=1e-16 档的舍入机理（B2 CH02-G22-B005 的 ULP 事实）===")
F = lambda t: np.sin(t*t) + np.exp(t)
x0 = 0.7
d1 = 2*x0*np.cos(x0*x0) + np.exp(x0)
h = 1e-16
xp, xm = x0 + h, x0 - h
print("  0.7+1e-16 == 0.7 ?", xp == x0, "  0.7-1e-16 == 0.7 ?", xm == x0, "  np.spacing(0.7) =", np.spacing(x0))
fw = (F(xp) - F(x0)) / h
cc = (F(xp) - F(xm)) / (2*h)
print(f"  前向值 = {fw!r}  误差 = {abs(fw-d1):.6e}   A1 题解记 3.24901871")
print(f"  中心值 = {cc!r}  误差 = {abs(cc-d1):.6e}   A1/A2 记 1.02857266 / 1.028573")
print("  判定：两稿数值可复现，不是错误；但\"前向差分为 0\"是两次函数值舍入到同一 double 的结果，不是 x+h=x")

print("\n=== C022 A1 中心差分表值与题解 4.2 符号（B1 丙-01/丙-02、B2 CH02-G22-B004）===")
cc01 = (F(0.8) - F(0.6)) / 0.2
print(f"  h=0.1 的中心差分值 = {cc01!r}   误差 = {cc01-d1:.6e}")
print("  判定：A1 结果表 3.2563 系 F'+|误差|（方向写反）；题解符号应为 +h²F'''/6 —— 与 B1、B2 同判，本组独立复算支持")

print("\n=== 附带复核 B2 §2.1 的两个探针值（n=11）===")
n = 11
xe = nodes_equi(n); xc = nodes_cheb(n)
print("  x=0.95: 等距", bary_eval(xe, feq(xe), np.array([0.95]))[0],
      " Cheb", bary_eval(xc, feq(xc), np.array([0.95]))[0], " 真值", feq(0.95))
print("  B2 记录：等距 1.923631、Chebyshev 0.085535、真值 0.042440")

=== C018 A2 评价网格重合点（B2 CH02-G22-B008）===
  400 点中与训练点重合数 = 8  索引 = [0, 57, 114, 171, 228, 285, 342, 399]
  d= 3: 全部 400 点 max=0.0204473  剔除重合点(392点) max=0.0200912
  d=14: 全部 400 点 max=0.9688473  剔除重合点(392点) max=0.9688473
  B2 记录：d=3 由 0.0204473 变 0.0200912，d=14 仍约 0.968847 —— 与本组一致

=== C019 A1 "点越界应显式拒绝"与实现行为（B2 CH02-G22-B006）===
  节点(-1,0,1)、q=2x²-x+3：x=1.1 返回 4.32  q(1.1)= 4.32
  n= 8 等距 Runge 外推 x=1.5: p=486.835  f=0.0174672
  n=32 等距 Runge 外推 x=1.5: p=-4.97382e+11  f=0.0174672

=== C020 A1 三处 -h^\frac{3}{6} 排版（B2 CH02-G22-B003）===
  h=1/4 的几何因子 -h^3/6 = -1/384 = -0.0026041666666666665 ；B2 记 -1/384，一致: True
  A1 正文中该写法出现的行号 = [717, 800, 1394]

=== C021 h=1e-16 档的舍入机理（B2 CH02-G22-B005 的 ULP 事实）===
  0.7+1e-16 == 0.7 ? False   0.7-1e-16 == 0.7 ? False   np.spacing(0.7) = 1.1102230246251565e-16
  前向值 = 0.0  误差 = 3.249019e+00   A1 题解记 3.24901871
  中心值 = 2.220446049250313  误差 = 1.028573e+00   A1/A2 记 1.02857266 / 1.028573
  判定：两稿数值可复现，不是错误；但"前向差分为 0"是两次函数值舍入到同一 double 的结果，不是 x+h=x

=== 

## 结论与复测边界

- B2（G22）已于 2026-11-11 提交首轮材料（main `c328c14`），本组已交叉复核其 B001–B009（见单元 8 与批判报告 C018–C022），"B2 是否用了与 A 不同的 AI"一项已满足；本报告初稿时"截至 2026-11-10 无提交"的事实记录保留备查。
- 复测承诺：A 组修订并给出新 SHA 后，本组对 CH02-G6-C007、C008、C003、C013、C016、CH02-G6-C017 六条做定点复测，把结果追加到原问题，不改号。
- B2（G22）交付后，本组补做与其结论的对照（尤其"是否用了与 A 不同的 AI"）。
- 基准教材页码（印 26、29、39–40、46、61–64、73–75、92、106–107、113、128–130、134）已于 2026-11-10 由组员本人逐页复核并改签（本 Notebook 中两处页码提示文字随之更新，其余数值输出未作任何改动）；A2 引用的课件《第二章-插值法-线条漫画版.pdf》第 6、9–10、32、35 页经核与所引内容相符。
- 本 Notebook 未使用、未新增任何图片，全部证据为可检索的文本输出。
- 局限：本内核 Python 3.11.11 + NumPy 1.26.4 + mpmath 1.3.0，无 PyTorch；涉及训练末态与 LAPACK 途径的量只作量级与归一化比较。

## 单元 7：结论汇总与可复核性

下表是本笔记本各单元的输出对应关系；判断的完整表述与改进方案见[批判报告](C1组-批判报告.md)，推导过程见[手动推导](C1组-手动推导.md)。

In [32]:
rows = [
 ("C001", "A1 附录 A.3 “不会超过连续范数（除非漏掉峰值）”", "单元1-网格加密码", "已确认：网格最大误差恒 ≤ 连续上确界，括注方向相反；漏峰只会偏小"),
 ("C002", "A1 附录 B.5 离散正交“只有在……才有”", "单元2-Gauss码", "已确认：过强；还须指定次数范围与权，Gauss–Legendre 节点+权即反例"),
 ("C003", "√x 只证下界即断言阶 3/2 与比值 0.354", "单元3-√x码", "已确认：证明缺口；本组补上界 0.22109·h^{3/2} 并实测常数 0.2078862 = -ζ(-1/2)"),
 ("C004", "A1 §2.4.2 用 |f|≈3.25、M2≈2.86 而未说明含义", "单元4-最优步长码", "已确认：3.25 是 |F'(0.7)|，A1 自身 JSON 记 F=2.4843786；M2/M3 是点值非上确界"),
 ("C005", "A1 §2.1 用 Lebesgue 指数增长解释 Runge 却声明“不证”，并把 5.91e-9 只记作“量级可信”", "单元1-Λ码+高精度码", "已确认：本组实测 Λ_32=2.431e7，Λ·ε=5.40e-9 与 5.91e-9 相差 9.6%，可作定量证据；该点 float64 仅剩约 8.2 位有效数字"),
 ("C006", "“该换的是节点”被写成唯一对策", "单元1-分段线性码", "已确认：同节点分段线性在 33 点已达 0.0207（理论界 0.0244），基准教材 §2.5 即以分段低次为对策"),
 ("C007", "A1 正文 Simpson h/6 与自家题解 3.2 的 h/3 相差一倍", "单元3-Simpson码", "已确认：正文错误、题解正确；修复只需与同稿题解对齐"),
 ("C008", "A1 “区间扩大 100 倍”", "单元3-缩放码", "已确认：实测 1000.0（式(2.3.5) 含 (b-a)h²，N 不变时两者同时放大）"),
 ("C009", "A1 §2.5 “最佳常数（约 0.64）”基线未定义", "单元2-基线码", "已确认：只在 mean|·|（=2/π=0.63662）下成立，与本章其余处用的 max/MSE 口径不一致"),
 ("C010", "A1 结果表“反向 Graph 误差 1.1e-15”", "单元4-符号码", "已确认：实测 1 ulp = 4.441e-16，与其自身 JSON 不符"),
 ("C011", "A1 “矩形公式作为一阶对照”", "单元3-检索码", "已确认：脚本无任何矩形/中点求积；本组另记录一次假阳性检索（directly 含 rect）"),
 ("C012", "A2 的系数差 4.152 与 d=14 残差 1.028e-7", "单元2-d9d14码", "已确认（机理）：4.152 属 d=9 分支且未标次数；换成“公共正确位数”后 A2/B1/本组三者一致（≈3.5–3.8 位）"),
 ("C013", "A1 例 2.5.1 / A2 式(2.4-5) 只给函数族反例", "单元5-气泡码", "已确认（推论缺口）：本组给出“训练损失对扰动不敏感”的证明与实测；“扰动属于所用网络类”一步仍条件不足"),
 ("C014", "A1 定理 2.5.3 常数未谈锐性、无一般区间形式", "单元5-锐性码", "已确认（改进）：等号族使 1/2 不可改进；一般区间为 η+M(b-a)²/8"),
 ("C015", "A1 §2.4/A2 §2.3 的“扫描中最小误差”被当作最优步长证据", "单元4-高精度码", "已确认：本组 60 位复核显示细网格最低点处的截断误差比 float64 总误差大 2–3 个数量级，低谷来自偶然抵消"),
 ("C016", "A1 参考文献表 20 行“本组人工核对”全为未核对，版次年未核实", "报告 §6", "已确认：本组查阅原书核对 5 处页码（4 处正确、1 处需细化），补全 2008.12 第1次印刷与 ISBN；A2 明确未查原书"),
]

rows.append(("C018", "A2 评价网格含 8 个训练点", "单元8", "已确认（P2）：400 点中恰 8 个重合；d=3 误差 0.0204473→0.0200912，d=14 不变 0.9688473"))
rows.append(("C019", "A1 『点越界应显式拒绝』与实现不符", "单元8", "已确认（P2）：x=1.1 返回 4.32=q(1.1)，x=1.5 在 n=8/32 时为 486.835 / -4.97e11；声明与实现不一致"))
rows.append(("C020", "A1 三处 -h^frac{3}{6} 排版错误（分母被放进指数）", "单元8", "已确认（P2）：行 717/800/1394；h=1/4 时正确几何因子为 -1/384，渲染后幂次由 3 变 1/2"))
rows.append(("C021", "h=1e-16 档『前向差分为 0』的机理", "单元8", "已确认（P3，澄清非错误）：x+h≠x 但两次函数值舍入到同一 double，前向误差=|F'|=3.249019、中心 1.028573，与两稿一致"))
rows.append(("C022", "A1 中心差分表值 3.2563 与题解 4.2 符号", "单元8", "已确认（P1）：实测 3.2417166809463027；D_h^0F-F'=+h²F'''/6，与 B1 丙-01/丙-02、B2 B004 同向"))
print("复核结论一览（编号与批判报告一致）")
for r in rows:
    print(f"  CH02-G6-{r[0]} | {r[1]}\n        证据: {r[2]} | {r[3]}")
print("\n本笔记本内所有数值均由本单元上方的真实执行输出支撑；无手写结论。")

复核结论一览（编号与批判报告一致）
  CH02-G6-C001 | A1 附录 A.3 “不会超过连续范数（除非漏掉峰值）”
        证据: 单元1-网格加密码 | 已确认：网格最大误差恒 ≤ 连续上确界，括注方向相反；漏峰只会偏小
  CH02-G6-C002 | A1 附录 B.5 离散正交“只有在……才有”
        证据: 单元2-Gauss码 | 已确认：过强；还须指定次数范围与权，Gauss–Legendre 节点+权即反例
  CH02-G6-C003 | √x 只证下界即断言阶 3/2 与比值 0.354
        证据: 单元3-√x码 | 已确认：证明缺口；本组补上界 0.22109·h^{3/2} 并实测常数 0.2078862 = -ζ(-1/2)
  CH02-G6-C004 | A1 §2.4.2 用 |f|≈3.25、M2≈2.86 而未说明含义
        证据: 单元4-最优步长码 | 已确认：3.25 是 |F'(0.7)|，A1 自身 JSON 记 F=2.4843786；M2/M3 是点值非上确界
  CH02-G6-C005 | A1 §2.1 用 Lebesgue 指数增长解释 Runge 却声明“不证”，并把 5.91e-9 只记作“量级可信”
        证据: 单元1-Λ码+高精度码 | 已确认：本组实测 Λ_32=2.431e7，Λ·ε=5.40e-9 与 5.91e-9 相差 9.6%，可作定量证据；该点 float64 仅剩约 8.2 位有效数字
  CH02-G6-C006 | “该换的是节点”被写成唯一对策
        证据: 单元1-分段线性码 | 已确认：同节点分段线性在 33 点已达 0.0207（理论界 0.0244），基准教材 §2.5 即以分段低次为对策
  CH02-G6-C007 | A1 正文 Simpson h/6 与自家题解 3.2 的 h/3 相差一倍
        证据: 单元3-Simpson码 | 已确认：正文错误、题解正确；修复只需与同稿题解对齐
  CH02-G6-C008 | A1 “区间扩大 100 倍”
        证据: 单元3-缩放码 | 已确认：实测 1000.0（式(2.3.5) 含 (b-a)h²，N 不变时两者同时放